# Milestone 2: a verified bounding box for every photo
**IE7615 Deep Learning, Group 7.** One notebook for the whole labelling pipeline. Every photo of the 73 objects gets one box around its object, and **every box is checked by a person** before it is used.

| Stage | Method | Who runs it |
|---|---|---|
| 1 | **Text prompt**: OWLv2 finds the object from its name | Claude Code |
| 1b | **Visual prompt**: OWLv2 finds the object from 3 example crops of it | Claude Code |
| 2 | **Self-trained detector**: YOLO trained on photos where 1 and 1b agree (training split only), then run on every photo | Claude Code |
| 3 | **Vote**: 3 boxes agree = *strong*, 2 agree = *auto*, otherwise *review* | Claude Code |
| 4 | **Review tool**: *verify* (grid, click wrong boxes) and *fix* (pick or draw a box) | **People** |
| 5 | **Master file**: final verified boxes, completeness check, accuracy of the automatic boxes | Claude Code |

**How to run.** The definition cells only define code. The **Run** cells at the bottom do the work, and only the steps listed in `M2_RUN` run, so each step can be run, checked, then followed by the next:
```
M2_RUN = stage1_test | stage1 | stage1b_test | stage1b | sheets | stage23_quick | stage23 | master | master_strict
```
Headless (Claude Code), PowerShell:
```
$env:M2_RUN="stage1_test"; jupyter nbconvert --to notebook --execute M2_Boxes.ipynb --output M2_Boxes_run.ipynb --ExecutePreprocessor.timeout=-1
```
**Safety (every stage):** a log file per run in `milestone2_outputs/logs/`, checkpoint after every batch (re-running the same step resumes), run fingerprints (results from different inputs or settings are archived to `_stale/`, never mixed), per-photo error handling, GPU out-of-memory fallback, atomic file writes.

## Configuration

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = r"C:\Users\csk23\Documents\Zanthosh\Northeastern\SEMESTERs\4th Sem\Deep Learning\PROJECT - 1"

# Which steps the Run cells execute (comma-separated). Headless: set the environment variable M2_RUN.
# Interactive: write them here, e.g. RUN = {"stage1_test"}.
RUN = {s.strip() for s in os.environ.get("M2_RUN", "").split(",") if s.strip()}

# Review (stage 4, people only): fill these in and run the "Review" cell. Leave REVIEWER empty otherwise.
# (These can also be set from the command line: M2_REVIEWER, M2_REVIEW_MODE, M2_REVIEW_CLASSES.)
REVIEWER       = os.environ.get("M2_REVIEWER", "")              # your initials, e.g. "SC"
REVIEW_MODE    = os.environ.get("M2_REVIEW_MODE", "verify")     # "exemplar" (3 example boxes per class, before stage 1b), or "verify" then "fix"
REVIEW_CLASSES = os.environ.get("M2_REVIEW_CLASSES", "")        # your share, e.g. "OBJ001-OBJ018"; for "exemplar" e.g. "OBJ040,OBJ043"
REVIEW_SPLITS  = os.environ.get("M2_REVIEW_SPLITS", "test")     # which splits to review: "test" (default), "test,val", or "" for all
REVIEW_SAMPLE  = int(os.environ.get("M2_REVIEW_SAMPLE", "0") or 0)   # verify mode: random sample of N auto boxes (0 = all)
REVIEW_REDO    = os.environ.get("M2_REVIEW_REDO", "") == "1"         # verify mode: show every photo of the classes again

VALID = {"stage1_test", "stage1", "stage1b_test", "stage1b", "sheets", "stage23_quick", "stage23", "master", "master_strict"}
assert RUN <= VALID, f"Unknown step(s) in M2_RUN: {RUN - VALID}"
print("Project:", PROJECT_DIR, "| steps to run:", sorted(RUN) or "none (definitions only)")

## Shared settings and helpers
Paths, the exclusion list (OBJ054 screenshot and the five OBJ004 background shots without the toy car), logging, safe file writing, image loading (same rule as Milestone 1) and box overlap.

In [ ]:
def _common():
    import hashlib, json, os, sys
    from datetime import datetime
    from pathlib import Path

    import numpy as np
    import pandas as pd
    from PIL import Image, ImageOps

    # ----------------------------------------------------------------------------------------------------------
    # Paths
    # ----------------------------------------------------------------------------------------------------------
    PROJECT    = Path(PROJECT_DIR)                # set in the Configuration cell
    DATA_ROOT  = PROJECT / "Dataset - Updated"
    M1_DIR     = PROJECT / "milestone1_outputs_v2"
    CLEAN_LIST = M1_DIR / "clean_dataset.csv"        # 6,942 cleaned photos (Milestone 1, V2 run)
    SPLIT_LIST = M1_DIR / "split.csv"                # Milestone 1 split: train / val / test
    OBJECT_LIST = PROJECT / "object_list.csv"        # course Object List (Google Sheets -> CSV)
    OUT_DIR    = PROJECT / "milestone2_outputs"
    TEXT_BOXES = OUT_DIR / "boxes_auto.csv"          # stage 1  (text prompt)
    VIS_BOXES  = OUT_DIR / "boxes_visual.csv"        # stage 1b (visual prompt)
    REVIEW_DIR = OUT_DIR / "review"                  # stage 4  (one CSV per reviewer)

    IMG_SIZE = 224                                    # every box in this pipeline is in the photo's 224x224 frame

    # Photos that are NOT photos of their object (never boxed, never used in Milestone 2):
    EXCLUDE = {
        "OBJ054_017.jpg",                                         # phone screenshot with personal details
        "OBJ004_007.jpg", "OBJ004_014.jpg", "OBJ004_023.jpg",     # OBJ004 background shots without the toy car
        "OBJ004_030.jpg", "OBJ004_073.jpg",                       # (identified by eye; OBJ004 never tagged them "_bg_")
    }


    # ----------------------------------------------------------------------------------------------------------
    # Logging: every print goes to the console AND a log file
    # ----------------------------------------------------------------------------------------------------------
    class Tee:
        def __init__(self, stream, fh): self.stream, self.fh = stream, fh
        def write(self, s):
            self.stream.write(s)
            try: self.fh.write(s); self.fh.flush()
            except Exception: pass
        def flush(self): self.stream.flush()
        def __getattr__(self, name): return getattr(self.stream, name)

    def _base_stream(s):
        """The real output stream under any logging wrappers (so logs never nest and output stays in the notebook)."""
        while hasattr(s, "fh") and hasattr(s, "stream"): s = s.stream
        return s

    def start_logging(stage, tag=""):
        log_dir = OUT_DIR / "logs"; log_dir.mkdir(parents=True, exist_ok=True)
        path = log_dir / f"{stage}{tag}_{datetime.now():%Y%m%d_%H%M%S}.log"
        fh = open(path, "a", encoding="utf-8")
        sys.stdout, sys.stderr = Tee(_base_stream(sys.stdout), fh), Tee(_base_stream(sys.stderr), fh)
        return path

    def stop_logging():
        for s in (sys.stdout, sys.stderr):
            if hasattr(s, "fh"):
                try: s.fh.close()
                except Exception: pass
        sys.stdout, sys.stderr = _base_stream(sys.stdout), _base_stream(sys.stderr)

    def now(): return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    def step(msg): print(f"\n[{now()}] ===== {msg} =====", flush=True)


    # ----------------------------------------------------------------------------------------------------------
    # Safe file writing and fingerprints
    # ----------------------------------------------------------------------------------------------------------
    def atomic_write_text(path, text):
        path = Path(path); tmp = path.with_name(path.name + ".tmp")
        tmp.write_text(text, encoding="utf-8"); os.replace(tmp, path)

    def atomic_to_csv(df, path):
        path = Path(path); tmp = path.with_name(path.name + ".tmp")
        df.to_csv(tmp, index=False); os.replace(tmp, path)

    def atomic_save_image(img, path, **kw):
        path = Path(path); tmp = path.with_name(path.stem + ".tmp" + path.suffix)
        img.save(tmp, **kw); os.replace(tmp, path)

    def file_md5(path):
        h = hashlib.md5()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
        return h.hexdigest()

    def fingerprint(obj):
        return hashlib.md5(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:12]


    # ----------------------------------------------------------------------------------------------------------
    # Images and boxes
    # ----------------------------------------------------------------------------------------------------------
    def load_224(path):
        """Same rule as Milestone 1 and stages 1/1b: fix rotation (EXIF), RGB, squash (never crop) to 224x224."""
        with Image.open(path) as im:
            im.draft("RGB", (IMG_SIZE * 2, IMG_SIZE * 2))
            im = ImageOps.exif_transpose(im).convert("RGB")
            return im.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BICUBIC)

    def iou(a, b):
        """IoU of two boxes (x1, y1, x2, y2); NaN if either box is missing."""
        if a is None or b is None or any(pd.isna(v) for v in list(a) + list(b)): return np.nan
        iw, ih = min(a[2], b[2]) - max(a[0], b[0]), min(a[3], b[3]) - max(a[1], b[1])
        if iw <= 0 or ih <= 0: return 0.0
        inter = iw * ih
        union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
        return float(inter / union) if union > 0 else 0.0

    def area_frac(b):
        return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1]) / (IMG_SIZE * IMG_SIZE)

    def plausible(b, min_area=0.01, max_area=0.97):
        return b is not None and not any(pd.isna(v) for v in b) and min_area <= area_frac(b) <= max_area

    def photo_table():
        """All photos that need a box: cleaned M1 photos minus EXCLUDE, with split and path."""
        clean = pd.read_csv(CLEAN_LIST)
        split = pd.read_csv(SPLIT_LIST)[["fname", "split"]]
        df = clean.rename(columns={"class_name": "object_id"}).merge(split, on="fname", how="left")
        df = df[~df.fname.isin(EXCLUDE)].reset_index(drop=True)
        df["path"] = [str(DATA_ROOT / f"images_{c}" / f) for c, f in zip(df.object_id, df.fname)]
        if df.split.isna().any():
            raise ValueError(f"{int(df.split.isna().sum())} photos have no split in {SPLIT_LIST}")
        return df[["fname", "object_id", "split", "path"]]

    import types as _t
    return _t.SimpleNamespace(**{k: v for k, v in locals().items() if not k.startswith('__')})

C = _common()
print('C: shared settings and helpers ready')

## Stage 1: text-prompt boxes
```
Milestone 2, step 1: automatic bounding boxes for every cleaned single-object photo.

WHAT: For each cleaned Milestone 1 photo (all splits), the open-vocabulary detector OWLv2 is prompted with the
      photo's object name ("a photo of a portable keyboard", ...) and returns the box around that object.
      Boxes are stored once, per photo, in the photo's 224x224 frame, so they can be reused for any split,
      composite layout or later milestone.

SAFETY AND TRACEABILITY (same standard as Milestone 1):
  - Logging:     everything printed also goes to milestone2_outputs/logs/step1_<timestamp>.log, together with
                 the software versions, GPU, every setting, and fingerprints of the input files.
  - Checkpoint:  results are appended to the CSV after every batch (flushed to disk), and progress.json shows
                 live progress. If the run stops for any reason, re-running the same command resumes.
  - Fingerprint: every result file is stamped with a fingerprint of the inputs + settings. If the inputs or
                 settings change, old results are moved to _stale/ instead of being silently mixed in.
  - Robustness:  an unreadable photo is logged in failed_photos.csv and skipped (the run continues); a GPU
                 out-of-memory error retries the batch with a smaller batch size.
  - Atomic writes: summaries and previews are written to a temp file and renamed, so a crash never leaves a
                 half-written file.

OUTPUTS (in PROJECT - 1/milestone2_outputs/):
  boxes_auto.csv           one row per photo: file, Object ID, label, box (x1,y1,x2,y2), score, prompt, flag
  boxes_auto.meta.json     fingerprint and settings that produced boxes_auto.csv
  failed_photos.csv        photos that could not be processed (normally empty)
  progress.json            live progress while running
  run_summary.json         final counts, flag rates, timing
  box_summary.txt          human-readable summary (flags overall and per class)
  box_previews/<ID>.jpg    one grid per Object ID with boxes drawn (green = ok, red = flagged, flagged first)
  logs/step1_<time>.log    full log of every run
(With --limit N, every file name gets a "_test" suffix so a test never mixes with the full run.)
```

In [ ]:
def _stage1():
    import argparse, csv, hashlib, json, math, os, platform, re, shutil, sys, time, traceback
    from datetime import datetime
    from pathlib import Path

    import numpy as np
    import pandas as pd
    from PIL import Image, ImageOps, ImageDraw

    SCRIPT_VERSION = "m2-step1-v2"

    # ----------------------------------------------------------------------------------------------------------
    # Settings (changing any of these changes the run fingerprint, so old results are archived, not reused)
    # ----------------------------------------------------------------------------------------------------------
    PROJECT = C.PROJECT     # shared setting (Configuration cell)
    DATA_ROOT = C.DATA_ROOT   # shared setting (Configuration cell)
    CLEAN_LIST = C.CLEAN_LIST  # shared setting (Configuration cell)
    OBJECT_LIST = C.OBJECT_LIST # shared setting (Configuration cell)
    OUT_DIR = C.OUT_DIR     # shared setting (Configuration cell)
    IMG_SIZE = C.IMG_SIZE    # shared setting (Configuration cell)
    MODEL_ID    = "google/owlv2-base-patch16-ensemble"
    BATCH       = 4          # measured on the RTX 5070 Laptop: 4 = ~8 photos/s; 8+ spills out of GPU memory (very slow)
    EXCLUDE = C.EXCLUDE     # shared setting (Configuration cell)
    LOW_SCORE, MIN_AREA, MAX_AREA = 0.15, 0.01, 0.97                          # quality-flag thresholds
    FIELDS = ["fname", "object_id", "label", "path", "x1", "y1", "x2", "y2", "score", "prompt", "flag"]


    # ==========================================================================================================
    # Logging: every print goes to the console AND the log file
    # ==========================================================================================================
    class Tee:
        def __init__(self, stream, fh): self.stream, self.fh = stream, fh
        def write(self, s):
            self.stream.write(s)
            try: self.fh.write(s); self.fh.flush()
            except Exception: pass
        def flush(self): self.stream.flush()
        def __getattr__(self, name): return getattr(self.stream, name)

    def start_logging(out_dir, tag):
        log_dir = out_dir / "logs"; log_dir.mkdir(parents=True, exist_ok=True)
        path = log_dir / f"step1{tag}_{datetime.now():%Y%m%d_%H%M%S}.log"
        fh = open(path, "a", encoding="utf-8")
        sys.stdout, sys.stderr = Tee(C._base_stream(sys.stdout), fh), Tee(C._base_stream(sys.stderr), fh)
        return path

    def count_failed(path):
        """Rows in a failure log; a missing, empty (0-byte) or header-only file counts as 0 failures."""
        path = Path(path)
        if not path.exists() or path.stat().st_size == 0: return 0
        try: return len(pd.read_csv(path))
        except pd.errors.EmptyDataError: return 0

    def now(): return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    def step(msg): print(f"\n[{now()}] ===== {msg} =====")


    # ==========================================================================================================
    # Small utilities
    # ==========================================================================================================
    def file_md5(path):
        h = hashlib.md5()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
        return h.hexdigest()

    def atomic_write_text(path, text):
        tmp = path.with_name(path.name + ".tmp"); tmp.write_text(text, encoding="utf-8"); os.replace(tmp, path)

    def atomic_save_image(img, path, **kw):
        tmp = path.with_name(path.stem + ".tmp" + path.suffix); img.save(tmp, **kw); os.replace(tmp, path)

    def load_object_names(path):
        """Object List CSV -> {"OBJ012": "portable keyboard", ...}; detects the ID and name columns automatically."""
        if not path.exists():
            raise FileNotFoundError(f"Missing {path}. Export the course Object List (Google Sheets: File > Download > CSV).")
        t = pd.read_csv(path, dtype=str).fillna("")
        norm = lambda s: re.sub(r"\s+", "", s).upper()
        id_col = max(t.columns, key=lambda c: t[c].map(lambda v: bool(re.fullmatch(r"OBJ\d+", norm(v)))).mean())
        cands = [c for c in t.columns if c != id_col and re.search(r"object|item|name|description", c, re.I)
                 and not re.search(r"\bid\b|student|member|group|email|team|owner|link|url", c, re.I)]
        if not cands:
            raise ValueError(f"Could not find the object-name column in {path}. Columns: {list(t.columns)}")
        name_col, names = cands[0], {}
        for _, r in t.iterrows():
            m = re.fullmatch(r"OBJ0*(\d+)", norm(r[id_col]))
            if m and r[name_col].strip(): names[f"OBJ{int(m.group(1)):03d}"] = r[name_col].strip()
        print(f"Object names: {len(names)} read (ID column '{id_col}', name column '{name_col}'), "
              f"e.g. {dict(list(names.items())[:3])}")
        return names

    def prompts_for(name):
        n = name.strip().rstrip(".").lower()
        return [f"a photo of a {n}", f"a {n}", n]

    def load_224(path):
        """Same loading as Milestone 1: fix phone rotation (EXIF), RGB, squash (never crop) to 224x224."""
        with Image.open(path) as im:
            im.draft("RGB", (IMG_SIZE * 2, IMG_SIZE * 2))
            im = ImageOps.exif_transpose(im).convert("RGB")
            return im.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BICUBIC)

    def best_box(logits, boxes):
        """logits: (patches, prompts); boxes: (patches, 4) normalised cx,cy,w,h -> best box in 224-pixel coordinates."""
        scores = 1.0 / (1.0 + np.exp(-logits))
        p, q = np.unravel_index(np.argmax(scores), scores.shape)
        cx, cy, w, h = boxes[p]
        xy = [(cx - w / 2) * IMG_SIZE, (cy - h / 2) * IMG_SIZE, (cx + w / 2) * IMG_SIZE, (cy + h / 2) * IMG_SIZE]
        return tuple(float(np.clip(v, 0, IMG_SIZE)) for v in xy), float(scores[p, q]), int(q)

    def flag_for(box, score):
        x1, y1, x2, y2 = box
        area = max(0.0, x2 - x1) * max(0.0, y2 - y1) / (IMG_SIZE * IMG_SIZE)
        if score < LOW_SCORE: return "low_score"
        if area < MIN_AREA:   return "too_small"
        if area > MAX_AREA:   return "whole_image"
        return ""

    def run_fingerprint(clean_md5, names_md5, n_photos, limit):
        key = dict(version=SCRIPT_VERSION, model=MODEL_ID, img=IMG_SIZE, clean=clean_md5, names=names_md5,
                   exclude=sorted(EXCLUDE), thresholds=[LOW_SCORE, MIN_AREA, MAX_AREA], n=n_photos, limit=limit)
        return hashlib.md5(json.dumps(key, sort_keys=True).encode()).hexdigest()[:12], key


    # ==========================================================================================================
    # Checkpoint handling
    # ==========================================================================================================
    def read_existing(csv_path):
        """Read results written so far. A crash can leave a half-written last line: such rows and duplicates are dropped."""
        if not csv_path.exists() or csv_path.stat().st_size == 0: return pd.DataFrame(columns=FIELDS)
        df = pd.read_csv(csv_path, dtype={"flag": str}, on_bad_lines="skip", engine="python")
        df = df.dropna(subset=["fname", "x1", "y1", "x2", "y2", "score"])
        df = df.drop_duplicates("fname", keep="last")
        df["flag"] = df["flag"].fillna("")
        return df

    def prepare_checkpoint(out_csv, meta_path, sig, key):
        """Resume only if the existing results were produced by exactly the same inputs and settings."""
        if out_csv.exists():
            old = json.loads(meta_path.read_text(encoding="utf-8")).get("sig") if meta_path.exists() else None
            if old != sig:
                dst = out_csv.parent / "_stale" / datetime.now().strftime("%Y%m%d_%H%M%S"); dst.mkdir(parents=True)
                for f in (out_csv, meta_path):
                    if f.exists(): shutil.move(str(f), str(dst / f.name))
                print(f"!! Existing results were made with different inputs/settings -> moved to {dst}; starting fresh")
            else:
                df = read_existing(out_csv)
                tmp = out_csv.with_name(out_csv.name + ".tmp"); df[FIELDS].to_csv(tmp, index=False); os.replace(tmp, out_csv)
                print(f"Resuming from checkpoint: {len(df)} photos already done")
        atomic_write_text(meta_path, json.dumps(dict(sig=sig, settings=key, created=now()), indent=2))


    # ==========================================================================================================
    # Core loop (model-independent, so it can be tested without a GPU)
    # ==========================================================================================================
    def process(todo, names, predict_fn, out_csv, failed_csv, progress_path, batch_size=BATCH):
        """predict_fn(list_of_PIL, list_of_prompt_lists) -> (logits[B,P,Q], boxes[B,P,4]) as numpy arrays."""
        new_file = not out_csv.exists() or out_csv.stat().st_size == 0
        fh = open(out_csv, "a", newline="", encoding="utf-8"); w = csv.DictWriter(fh, fieldnames=FIELDS)
        if new_file: w.writeheader(); fh.flush()
        failed_new = not failed_csv.exists() or failed_csv.stat().st_size == 0     # missing or empty -> write header
        ff = open(failed_csv, "a", newline="", encoding="utf-8"); fw = csv.writer(ff)
        if failed_new: fw.writerow(["fname", "path", "error", "time"]); ff.flush(); os.fsync(ff.fileno())
        t0, done, n_fail, bs, s = time.time(), 0, 0, batch_size, 0
        try:
            while s < len(todo):
                batch = todo.iloc[s:s + bs]
                imgs, rows = [], []
                for _, r in batch.iterrows():                      # load each photo; an unreadable one is logged, not fatal
                    try: imgs.append(load_224(r.path)); rows.append(r)
                    except Exception as e:
                        fw.writerow([r.fname, r.path, repr(e), now()]); ff.flush(); n_fail += 1
                        print(f"  ! could not read {r.fname}: {e!r}")
                if rows:
                    texts = [prompts_for(names[r.class_name]) for r in rows]
                    try:
                        logits, boxes = predict_fn(imgs, texts)
                    except RuntimeError as e:                      # GPU out of memory -> retry this batch smaller
                        if "out of memory" in str(e).lower() and bs > 1:
                            bs = max(1, bs // 2); print(f"  ! GPU out of memory -> retrying with batch size {bs}")
                            try:
                                import torch; torch.cuda.empty_cache()
                            except Exception: pass
                            continue
                        raise
                    for k, r in enumerate(rows):
                        box, score, q = best_box(logits[k], boxes[k])
                        w.writerow(dict(fname=r.fname, object_id=r.class_name, label=int(r.label), path=r.path,
                                        x1=round(box[0], 1), y1=round(box[1], 1), x2=round(box[2], 1), y2=round(box[3], 1),
                                        score=round(score, 4), prompt=texts[k][q], flag=flag_for(box, score)))
                    fh.flush(); os.fsync(fh.fileno())               # checkpoint: this batch is safely on disk
                s += len(batch); done += len(rows)
                elapsed = time.time() - t0; rate = (done + n_fail) / max(elapsed, 1e-6)
                atomic_write_text(progress_path, json.dumps(dict(processed=s, of=len(todo), failed=n_fail, batch_size=bs,
                                    photos_per_s=round(rate, 2), eta_min=round((len(todo) - s) / max(rate, 1e-6) / 60, 1),
                                    updated=now()), indent=2))
                if (s // max(bs, 1)) % 20 == 0 or s >= len(todo):
                    print(f"  [{now()}] {s}/{len(todo)} photos | {rate:.1f}/s | failed {n_fail} | "
                          f"~{(len(todo) - s) / max(rate, 1e-6) / 60:.1f} min left")
        finally:
            fh.close(); ff.close()
        return done, n_fail, time.time() - t0


    def save_previews(res, out_dir, per_class=12):
        out_dir.mkdir(parents=True, exist_ok=True)
        for cls, g in res.groupby("object_id"):
            g = pd.concat([g[g.flag != ""], g[g.flag == ""]]).head(per_class)
            cols = 6; rows_n = math.ceil(len(g) / cols); cell = IMG_SIZE + 24
            sheet = Image.new("RGB", (cols * cell, rows_n * cell), "white"); d = ImageDraw.Draw(sheet)
            for k, (_, r) in enumerate(g.iterrows()):
                try: im = load_224(r.path)
                except Exception: continue
                colour = (220, 30, 30) if r.flag else (30, 200, 60)
                ImageDraw.Draw(im).rectangle([r.x1, r.y1, r.x2, r.y2], outline=colour, width=3)
                x0, y0 = (k % cols) * cell, (k // cols) * cell
                sheet.paste(im, (x0 + 12, y0 + 20))
                d.text((x0 + 12, y0 + 4), f"{str(r.fname)[:-4]}  {r.score:.2f} {r.flag}", fill=colour if r.flag else (0, 0, 0))
            atomic_save_image(sheet, out_dir / f"{cls}.jpg", quality=90)


    def summarise(res, n_expected, n_failed, elapsed, sig, out_txt, out_json):
        flagged = res.flag != ""
        per_cls = res.assign(f=flagged).groupby("object_id").f.mean().sort_values(ascending=False)
        complete = len(res) + n_failed >= n_expected
        summary = dict(sig=sig, finished=now(), complete=bool(complete), photos_expected=int(n_expected),
                       photos_with_box=int(len(res)), photos_failed=int(n_failed),
                       flagged=int(flagged.sum()), flagged_pct=round(100 * float(flagged.mean()), 2) if len(res) else 0.0,
                       flag_types={k: int(v) for k, v in res.flag.replace("", "ok").value_counts().items()},
                       median_score=round(float(res.score.median()), 4) if len(res) else None,
                       runtime_min_this_run=round(elapsed / 60, 2),
                       most_flagged_classes={k: round(float(v), 3) for k, v in per_cls.head(15).items()})
        atomic_write_text(out_json, json.dumps(summary, indent=2))
        lines = [f"Run fingerprint: {sig}   finished {summary['finished']}",
                 f"Complete: {'YES' if complete else 'NO (re-run the same cell to resume)'}",
                 f"Photos expected: {n_expected} | with a box: {len(res)} | failed: {n_failed}",
                 f"Flagged: {summary['flagged']} ({summary['flagged_pct']}%)   types: {summary['flag_types']}",
                 f"Median confidence: {summary['median_score']}", "",
                 "Classes with the most flagged boxes:"] + [f"  {c}: {v:.0%}" for c, v in per_cls.head(15).items()]
        atomic_write_text(out_txt, "\n".join(lines))
        print("\n".join(lines))
        return summary


    # ==========================================================================================================
    # Main
    # ==========================================================================================================
    def main(limit=0):
        import types as _t
        args = _t.SimpleNamespace(**dict(locals()))
        tag = "_test" if args.limit else ""
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        log_path = start_logging(OUT_DIR, tag)
        out_csv, meta = OUT_DIR / f"boxes_auto{tag}.csv", OUT_DIR / f"boxes_auto{tag}.meta.json"
        failed_csv, progress = OUT_DIR / f"failed_photos{tag}.csv", OUT_DIR / f"progress{tag}.json"

        try:
            step(f"Step 1: automatic boxes ({SCRIPT_VERSION})")
            print(f"Log file: {log_path}")
            import torch, transformers
            from transformers import Owlv2Processor, Owlv2ForObjectDetection
            gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
            print(f"Python {platform.python_version()} | torch {torch.__version__} | transformers {transformers.__version__} "
                  f"| CUDA available {torch.cuda.is_available()} | GPU {gpu}")

            step("Inputs")
            for p in (DATA_ROOT, CLEAN_LIST, OBJECT_LIST):
                if not p.exists(): raise FileNotFoundError(f"Missing input: {p}")
            names = load_object_names(OBJECT_LIST)
            clean = pd.read_csv(CLEAN_LIST)
            print(f"clean_dataset.csv: {len(clean)} photos, {clean.class_name.nunique()} classes")
            clean = clean[~clean.fname.isin(EXCLUDE)].reset_index(drop=True)
            print(f"Excluded {sorted(EXCLUDE)} -> {len(clean)} photos")
            missing = sorted(set(clean.class_name) - set(names))
            if missing: raise ValueError(f"No object name for {missing} in {OBJECT_LIST}")
            clean["path"] = [str(DATA_ROOT / f"images_{c}" / f) for c, f in zip(clean.class_name, clean.fname)]
            not_found = [p for p in clean.path if not os.path.exists(p)]
            if not_found: raise FileNotFoundError(f"{len(not_found)} photos listed but not on disk, e.g. {not_found[:3]}")
            if args.limit: clean = clean.sample(n=min(args.limit, len(clean)), random_state=0).reset_index(drop=True)

            step("Checkpoint")
            sig, key = run_fingerprint(file_md5(CLEAN_LIST), file_md5(OBJECT_LIST), len(clean), args.limit)
            print(f"Run fingerprint: {sig}   settings: {json.dumps(key)}")
            prepare_checkpoint(out_csv, meta, sig, key)
            done_names = set(read_existing(out_csv).fname)
            todo = clean[~clean.fname.isin(done_names)].reset_index(drop=True)
            print(f"Already done: {len(done_names)} | to process now: {len(todo)}")

            elapsed, n_fail = 0.0, 0
            if len(todo):
                step(f"Loading {MODEL_ID}")
                device = "cuda" if torch.cuda.is_available() else "cpu"
                processor = Owlv2Processor.from_pretrained(MODEL_ID)
                model = Owlv2ForObjectDetection.from_pretrained(MODEL_ID).to(device).eval()
                if device == "cuda": model = model.half()

                @torch.no_grad()
                def predict(imgs, texts):
                    inp = processor(text=texts, images=imgs, return_tensors="pt").to(device)
                    if device == "cuda": inp["pixel_values"] = inp["pixel_values"].half()
                    out = model(**inp)
                    return out.logits.float().cpu().numpy(), out.pred_boxes.float().cpu().numpy()

                step("Detecting boxes")
                _, n_fail, elapsed = process(todo, names, predict, out_csv, failed_csv, progress)

            step("Summary and previews")
            res = read_existing(out_csv)
            res = res[res.fname.isin(set(clean.fname))]
            total_failed = count_failed(failed_csv)
            summarise(res, len(clean), total_failed, elapsed, sig, OUT_DIR / f"box_summary{tag}.txt", OUT_DIR / f"run_summary{tag}.json")
            save_previews(res, OUT_DIR / f"box_previews{tag}")
            print(f"\nOutputs in {OUT_DIR}\n[{now()}] DONE")
        except KeyboardInterrupt:
            print(f"\n[{now()}] Interrupted. Everything up to the last batch is saved; re-run the same cell to resume.")
            raise
        except Exception:
            print(f"\n[{now()}] FAILED:\n{traceback.format_exc()}")
            print("Results up to the last completed batch are saved; fix the error and re-run the same cell to resume.")
            raise

    import types as _t
    return _t.SimpleNamespace(**{k: v for k, v in locals().items() if not k.startswith('__')})

S1 = _stage1()
print('S1: stage 1 (text prompt) ready')

## Stage 1b: visual-prompt boxes
```
Milestone 2, step 1b: VISUAL-PROMPT boxes ("find this object", shown by a picture instead of a name).

WHAT: For every Object ID, the best boxed crops from TRAINING photos (from step 1's text-prompt boxes) are used as
      example images ("exemplars"). OWLv2's image-guided detection then searches each photo for the region that
      looks most like its own object's exemplars. The result is compared with step 1's text-prompt box:
        agree (IoU >= AGREE_IOU)  -> accepted automatically (two independent methods found the same box)
        disagree / one missing    -> listed for human review
WHY:  Text prompts fail when the detector does not know a name ("iWatch", "mouth guard case"). Each class is one
      physical object photographed ~100 times, so a few real crops of it describe it better than any word.

INPUTS : boxes_auto.csv (step 1), clean_dataset.csv and split.csv (Milestone 1 V2), the dataset folder.
OUTPUTS (milestone2_outputs/):
  exemplars.csv / exemplar_sheet.jpg     which crops were used as examples for each Object ID (check these by eye)
  boxes_visual.csv                       one row per photo: visual box, score, IoU with the text box, decision
  boxes_combined.csv                     the box to use per photo + decision: accept / review
  review_list.csv                        photos a person should check (with both boxes)
  visual_previews/<ID>.jpg               per class: green = text box, magenta = visual box
  visual_summary.txt, visual_run_summary.json, visual_progress.json, logs/step1b_<time>.log
Same safety standard as step 1: full log, per-batch checkpoint and resume, run fingerprint (stale results are
archived, never mixed), per-photo error handling, GPU out-of-memory fallback, atomic writes.
```

In [ ]:
def _stage1b():
    import argparse, csv, hashlib, json, math, os, platform, shutil, sys, time, traceback
    from datetime import datetime
    from pathlib import Path

    import numpy as np
    import pandas as pd
    from PIL import Image, ImageOps, ImageDraw

    SCRIPT_VERSION = "m2-step1b-v3"

    # ----------------------------------------------------------------------------------------------------------
    # Settings (changing any of these changes the run fingerprint, so old results are archived, not reused)
    # ----------------------------------------------------------------------------------------------------------
    PROJECT = C.PROJECT     # shared setting (Configuration cell)
    DATA_ROOT = C.DATA_ROOT   # shared setting (Configuration cell)
    M1_DIR = C.M1_DIR      # shared setting (Configuration cell)
    CLEAN_LIST = C.CLEAN_LIST  # shared setting (Configuration cell)
    SPLIT_LIST = C.SPLIT_LIST  # shared setting (Configuration cell)
    OUT_DIR = C.OUT_DIR     # shared setting (Configuration cell)
    TEXT_BOXES = C.TEXT_BOXES  # shared setting (Configuration cell)
    MANUAL_EXEMPLARS = OUT_DIR / "exemplars_manual.csv"    # hand-drawn exemplars (review tool, mode "exemplar"); override per class
    IMG_SIZE = C.IMG_SIZE    # shared setting (Configuration cell)
    MODEL_ID    = "google/owlv2-base-patch16-ensemble"
    EXCLUDE = C.EXCLUDE     # shared setting (Configuration cell)

    N_EXEMPLARS     = 3        # example crops per Object ID
    EXEMPLAR_MINSC  = 0.30     # an exemplar's text box must have at least this confidence and no quality flag
    EXEMPLAR_PAD    = 0.06     # margin around an exemplar crop (fraction of its size)
    AGREE_IOU       = 0.70     # text box and visual box overlapping at least this much = accepted
    TARGETS_PER_BATCH = 4      # photos per forward pass (measured: 4 fits the 8 GB GPU; 8+ spills, very slow)
    FALLBACK_PAIR_CHUNK = 2    # only if the fast path is unavailable: (photo, exemplar) pairs per pass
    MIN_AREA, MAX_AREA = 0.01, 0.97
    FIELDS = ["fname", "object_id", "split", "path", "vx1", "vy1", "vx2", "vy2", "vscore", "exemplar_used", "is_exemplar"]


    # ==========================================================================================================
    # Logging and small utilities (self-contained)
    # ==========================================================================================================
    class Tee:
        def __init__(self, stream, fh): self.stream, self.fh = stream, fh
        def write(self, s):
            self.stream.write(s)
            try: self.fh.write(s); self.fh.flush()
            except Exception: pass
        def flush(self): self.stream.flush()
        def __getattr__(self, name): return getattr(self.stream, name)

    def start_logging(out_dir, tag):
        (out_dir / "logs").mkdir(parents=True, exist_ok=True)
        path = out_dir / "logs" / f"step1b{tag}_{datetime.now():%Y%m%d_%H%M%S}.log"
        fh = open(path, "a", encoding="utf-8"); sys.stdout, sys.stderr = Tee(C._base_stream(sys.stdout), fh), Tee(C._base_stream(sys.stderr), fh)
        return path

    def count_failed(path):
        """Rows in a failure log; a missing, empty (0-byte) or header-only file counts as 0 failures."""
        path = Path(path)
        if not path.exists() or path.stat().st_size == 0: return 0
        try: return len(pd.read_csv(path))
        except pd.errors.EmptyDataError: return 0

    def now(): return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    def step(msg): print(f"\n[{now()}] ===== {msg} =====")

    def file_md5(path):
        h = hashlib.md5()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
        return h.hexdigest()

    def atomic_write_text(path, text):
        tmp = path.with_name(path.name + ".tmp"); tmp.write_text(text, encoding="utf-8"); os.replace(tmp, path)

    def atomic_to_csv(df, path):
        tmp = path.with_name(path.name + ".tmp"); df.to_csv(tmp, index=False); os.replace(tmp, path)

    def atomic_save_image(img, path, **kw):
        tmp = path.with_name(path.stem + ".tmp" + path.suffix); img.save(tmp, **kw); os.replace(tmp, path)

    def load_224(path):
        """Same loading as Milestone 1 and step 1: fix rotation (EXIF), RGB, squash (never crop) to 224x224."""
        with Image.open(path) as im:
            im.draft("RGB", (IMG_SIZE * 2, IMG_SIZE * 2))
            im = ImageOps.exif_transpose(im).convert("RGB")
            return im.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BICUBIC)

    def iou(a, b):
        if any(pd.isna(v) for v in list(a) + list(b)): return np.nan
        iw, ih = min(a[2], b[2]) - max(a[0], b[0]), min(a[3], b[3]) - max(a[1], b[1])
        if iw <= 0 or ih <= 0: return 0.0
        inter = iw * ih
        return float(inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter))

    def area_frac(b): return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1]) / (IMG_SIZE * IMG_SIZE)

    def best_box(logits, boxes):
        """logits: (patches, exemplars); boxes: (patches, 4) normalised cx,cy,w,h.
        Returns the best box over all patches and exemplars (224-px coords), its score, and which exemplar matched."""
        logits = np.asarray(logits, dtype=np.float64).reshape(len(boxes), -1)
        # Choose on the RAW logits: image-guided logits are large, and after the sigmoid many patches round to ~1.0,
        # which would make the choice arbitrary. The sigmoid is only used for the reported score.
        p, q = np.unravel_index(np.argmax(logits), logits.shape)
        score = float(1.0 / (1.0 + np.exp(-np.clip(logits[p, q], -50, 50))))
        cx, cy, w, h = boxes[p]
        xy = [(cx - w / 2) * IMG_SIZE, (cy - h / 2) * IMG_SIZE, (cx + w / 2) * IMG_SIZE, (cy + h / 2) * IMG_SIZE]
        return tuple(float(np.clip(v, 0, IMG_SIZE)) for v in xy), score, int(q)

    def same_scoring(l_fast, l_orig, tol=0.25):
        """Do two methods score a photo the same way? Pattern of logits over all patches/exemplars (correlation) AND
        the fast path's chosen patch is a top choice for the original method (logit gap within tol). Exact argmax
        equality is not required, because near-ties are common and tiny rounding differences can swap them."""
        a = np.asarray(l_fast, np.float64).ravel(); b = np.asarray(l_orig, np.float64).ravel()
        corr = float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else 1.0
        gap = float(b.max() - b[int(np.argmax(a))])
        return corr, gap, (corr >= 0.98 and gap <= tol)


    # ==========================================================================================================
    # Exemplars: the best text-prompt boxes from TRAINING photos, spread over different shots
    # ==========================================================================================================
    def choose_exemplars(text, split):
        tr = text.merge(split[["fname", "split"]], on="fname")
        tr = tr[(tr.split == "train") & (tr.flag.fillna("") == "") & (tr.score >= EXEMPLAR_MINSC)].copy()
        tr = tr.sort_values(["object_id", "score"], ascending=[True, False])
        picked = []
        for cls, g in tr.groupby("object_id"):
            chosen = []
            for _, r in g.iterrows():                       # prefer exemplars from different backgrounds (shot ranges)
                shot = int("".join(ch for ch in Path(r.fname).stem.split("_")[-1] if ch.isdigit()) or 0)
                if all(abs(shot - c[1]) >= 15 for c in chosen) or len(g) <= N_EXEMPLARS:
                    chosen.append((r, shot))
                if len(chosen) == N_EXEMPLARS: break
            if len(chosen) < N_EXEMPLARS:                   # not enough diversity: fill with the next best boxes
                used = {c[0].fname for c in chosen}
                for _, r in g.iterrows():
                    if r.fname not in used: chosen.append((r, -1)); used.add(r.fname)
                    if len(chosen) == N_EXEMPLARS: break
            picked += [dict(object_id=cls, fname=r.fname, path=r.path, x1=r.x1, y1=r.y1, x2=r.x2, y2=r.y2, score=r.score)
                       for r, _ in chosen]
        return pd.DataFrame(picked)

    def apply_manual_exemplars(ex, photos, path=None):
        """Replace a class's automatic exemplars with the hand-drawn ones (latest box per photo, up to N per class)."""
        path = Path(path or MANUAL_EXEMPLARS)
        if not path.exists() or path.stat().st_size == 0: return ex, []
        man = pd.read_csv(path).dropna(subset=["fname", "x1", "y1", "x2", "y2"])
        man = man.sort_values("time", kind="stable").drop_duplicates("fname", keep="last")
        man = man.merge(photos[["fname", "path"]], on="fname", how="inner")
        man = man.groupby("object_id").tail(N_EXEMPLARS)
        classes = sorted(man.object_id.unique())
        keep = ex[~ex.object_id.isin(classes)]
        add = man.assign(score=1.0)[["object_id", "fname", "path", "x1", "y1", "x2", "y2", "score"]]
        return pd.concat([keep, add], ignore_index=True).sort_values(["object_id", "fname"]).reset_index(drop=True), classes

    def crop_exemplar(row):
        im = load_224(row["path"])
        w, h = row["x2"] - row["x1"], row["y2"] - row["y1"]
        pad_w, pad_h = EXEMPLAR_PAD * w, EXEMPLAR_PAD * h
        box = (max(0, row["x1"] - pad_w), max(0, row["y1"] - pad_h), min(IMG_SIZE, row["x2"] + pad_w), min(IMG_SIZE, row["y2"] + pad_h))
        return im.crop(tuple(int(round(v)) for v in box))

    def exemplar_sheet(ex, path, names=None, manual=()):
        """One row per class: Object ID (+ name), then its exemplar crops, large enough to judge by eye.
        Split into pages of 25 classes (exemplar_sheet_1.jpg, ...) so each image stays readable."""
        classes = sorted(ex.object_id.unique()); cell, lab, per_page = 170, 190, 25
        path = Path(path); written = []
        for pg in range(0, len(classes), per_page):
            chunk = classes[pg:pg + per_page]
            sheet = Image.new("RGB", (lab + cell * N_EXEMPLARS, (cell + 18) * len(chunk)), "white"); d = ImageDraw.Draw(sheet)
            for i, cls in enumerate(chunk):
                y0 = i * (cell + 18)
                tag = " (hand-drawn)" if cls in manual else ""
                d.text((6, y0 + cell // 2 - 12), f"{cls}{tag}", fill=(0, 0, 0))
                if names and cls in names: d.text((6, y0 + cell // 2 + 4), str(names[cls])[:26], fill=(90, 90, 90))
                for j, (_, r) in enumerate(ex[ex.object_id == cls].iterrows()):
                    c = crop_exemplar(r); c.thumbnail((cell - 10, cell - 10))
                    sheet.paste(c, (lab + j * cell + 5, y0 + 5))
                    d.text((lab + j * cell + 5, y0 + cell - 2), f"{str(r.fname)[:-4]} {r.score:.2f}", fill=(0, 0, 0))
            out = path.with_name(f"{path.stem}_{pg // per_page + 1}{path.suffix}")
            atomic_save_image(sheet, out, quality=92); written.append(out.name)
        return written


    # ==========================================================================================================
    # Checkpoint handling and the model-independent core loop (testable without a GPU)
    # ==========================================================================================================
    def read_existing(path):
        if not path.exists() or path.stat().st_size == 0: return pd.DataFrame(columns=FIELDS)
        df = pd.read_csv(path, on_bad_lines="skip", engine="python")
        return df.dropna(subset=["fname", "vx1", "vy1", "vx2", "vy2", "vscore"]).drop_duplicates("fname", keep="last")

    def prepare_checkpoint(out_csv, meta, sig, key):
        if out_csv.exists():
            old = json.loads(meta.read_text(encoding="utf-8")).get("sig") if meta.exists() else None
            if old != sig:
                dst = out_csv.parent / "_stale" / datetime.now().strftime("%Y%m%d_%H%M%S"); dst.mkdir(parents=True)
                for f in (out_csv, meta):
                    if f.exists(): shutil.move(str(f), str(dst / f.name))
                print(f"!! Existing visual boxes were made with different inputs/settings -> moved to {dst}; starting fresh")
            else:
                df = read_existing(out_csv); atomic_to_csv(df[FIELDS], out_csv)
                print(f"Resuming from checkpoint: {len(df)} photos already done")
        atomic_write_text(meta, json.dumps(dict(sig=sig, settings=key, created=now()), indent=2))

    def process(todo, exemplar_crops, ex_names, predict_fn, out_csv, failed_csv, progress, per_batch=TARGETS_PER_BATCH):
        """predict_fn(targets: list[PIL], classes: list[str]) -> (logits[B, P, K], boxes[B, P, 4]):
        each photo scored against the K exemplars of its own class in ONE pass (K = exemplar count)."""
        new = not out_csv.exists() or out_csv.stat().st_size == 0
        fh = open(out_csv, "a", newline="", encoding="utf-8"); w = csv.DictWriter(fh, fieldnames=FIELDS)
        if new: w.writeheader(); fh.flush()
        fnew = not failed_csv.exists() or failed_csv.stat().st_size == 0           # missing or empty -> write header
        ff = open(failed_csv, "a", newline="", encoding="utf-8"); fw = csv.writer(ff)
        if fnew: fw.writerow(["fname", "path", "error", "time"]); ff.flush(); os.fsync(ff.fileno())
        t0, s, n_fail, bs = time.time(), 0, 0, per_batch
        try:
            while s < len(todo):
                batch = todo.iloc[s:s + bs]; rows, targets = [], []
                for _, r in batch.iterrows():
                    try: targets.append(load_224(r.path)); rows.append(r)
                    except Exception as e:
                        fw.writerow([r.fname, r.path, repr(e), now()]); ff.flush(); n_fail += 1
                        print(f"  ! could not read {r.fname}: {e!r}")
                if rows:
                    try:
                        logits, boxes = predict_fn(targets, [r.object_id for r in rows])
                    except RuntimeError as e:
                        if "out of memory" in str(e).lower() and bs > 1:
                            bs = max(1, bs // 2); print(f"  ! GPU out of memory -> retrying with {bs} photos per batch")
                            try:
                                import torch; torch.cuda.empty_cache()
                            except Exception: pass
                            continue
                        raise
                    for k, r in enumerate(rows):
                        n_ex = len(ex_names[r.object_id])
                        box, score, j = best_box(np.asarray(logits[k])[:, :n_ex], boxes[k])
                        w.writerow(dict(fname=r.fname, object_id=r.object_id, split=r.split, path=r.path,
                                        vx1=round(box[0], 1), vy1=round(box[1], 1), vx2=round(box[2], 1), vy2=round(box[3], 1),
                                        vscore=round(score, 4), exemplar_used=ex_names[r.object_id][j],
                                        is_exemplar=int(r.fname in ex_names[r.object_id])))
                    fh.flush(); os.fsync(fh.fileno())
                s += len(batch)
                rate = s / max(time.time() - t0, 1e-6)
                atomic_write_text(progress, json.dumps(dict(processed=s, of=len(todo), failed=n_fail, photos_per_batch=bs,
                                  photos_per_s=round(rate, 2), eta_min=round((len(todo) - s) / max(rate, 1e-6) / 60, 1), updated=now()), indent=2))
                if (s // max(bs, 1)) % 25 == 0 or s >= len(todo):
                    print(f"  [{now()}] {s}/{len(todo)} photos | {rate:.1f}/s | failed {n_fail} | ~{(len(todo) - s) / max(rate, 1e-6) / 60:.1f} min left")
        finally:
            fh.close(); ff.close()
        return n_fail, time.time() - t0


    # ==========================================================================================================
    # Combine text and visual boxes
    # ==========================================================================================================
    def combine(text, vis):
        df = vis.merge(text[["fname", "x1", "y1", "x2", "y2", "score", "flag"]].rename(columns={"score": "tscore", "flag": "tflag"}),
                       on="fname", how="left")
        df["tflag"] = df.tflag.fillna("no_text_box").astype(str).replace("nan", "")
        df["iou_text_visual"] = [iou((a, b, c, d), (e, f, g, h)) for a, b, c, d, e, f, g, h in
                                 zip(df.x1, df.y1, df.x2, df.y2, df.vx1, df.vy1, df.vx2, df.vy2)]
        v_ok = [MIN_AREA <= area_frac(b) <= MAX_AREA for b in zip(df.vx1, df.vy1, df.vx2, df.vy2)]
        agree = (df.iou_text_visual >= AGREE_IOU) & np.array(v_ok)
        df["decision"] = np.where(agree, "accept", "review")
        df["reason"] = np.select(
            [agree, df.tflag == "no_text_box", ~np.array(v_ok), df.iou_text_visual < AGREE_IOU],
            ["text and visual boxes agree", "no text box", "visual box too small or whole photo", "text and visual boxes disagree"],
            default="")
        # Box to use: the average of the two when they agree (reduces each method's own error); otherwise the visual
        # box is offered first for review, because it does not depend on the object's name.
        for a, (t, v) in {"bx1": ("x1", "vx1"), "by1": ("y1", "vy1"), "bx2": ("x2", "vx2"), "by2": ("y2", "vy2")}.items():
            df[a] = np.where(agree, (df[t] + df[v]) / 2, df[v]).round(1)
        return df


    def save_previews(df, out_dir, per_class=12):
        out_dir.mkdir(parents=True, exist_ok=True)
        for cls, g in df.groupby("object_id"):
            g = pd.concat([g[g.decision == "review"], g[g.decision == "accept"]]).head(per_class)
            cols = 6; cell = IMG_SIZE + 26; sheet = Image.new("RGB", (cols * cell, math.ceil(len(g) / cols) * cell), "white")
            d = ImageDraw.Draw(sheet)
            for k, (_, r) in enumerate(g.iterrows()):
                try: im = load_224(r.path)
                except Exception: continue
                dd = ImageDraw.Draw(im)
                if not pd.isna(r.x1): dd.rectangle([r.x1, r.y1, r.x2, r.y2], outline=(30, 200, 60), width=3)   # text box
                dd.rectangle([r.vx1, r.vy1, r.vx2, r.vy2], outline=(220, 0, 220), width=2)                      # visual box
                x0, y0 = (k % cols) * cell, (k // cols) * cell
                sheet.paste(im, (x0 + 13, y0 + 22))
                iou_txt = "-" if pd.isna(r.iou_text_visual) else f"{r.iou_text_visual:.2f}"
                d.text((x0 + 13, y0 + 4), f"{str(r.fname)[:-4]} IoU {iou_txt} {r.decision}",
                       fill=(200, 0, 0) if r.decision == "review" else (0, 0, 0))
            atomic_save_image(sheet, out_dir / f"{cls}.jpg", quality=90)


    def summarise(df, n_expected, n_failed, elapsed, sig, out_txt, out_json):
        acc = df.decision == "accept"
        per_cls = df.assign(r=~acc).groupby("object_id").r.mean().sort_values(ascending=False)
        complete = len(df) + n_failed >= n_expected
        by_split = df.groupby("split").decision.apply(lambda d: round(100 * float((d == "accept").mean()), 1)).to_dict()
        s = dict(sig=sig, finished=now(), complete=bool(complete), photos_expected=int(n_expected), photos_done=int(len(df)),
                 photos_failed=int(n_failed), accepted=int(acc.sum()), review=int((~acc).sum()),
                 accepted_pct=round(100 * float(acc.mean()), 1) if len(df) else 0.0, accepted_pct_by_split=by_split,
                 review_reasons=df[~acc].reason.value_counts().to_dict(),
                 median_iou_text_visual=round(float(df.iou_text_visual.median()), 3) if len(df) else None,
                 runtime_min_this_run=round(elapsed / 60, 2),
                 classes_needing_most_review={k: round(float(v), 3) for k, v in per_cls.head(15).items()})
        atomic_write_text(out_json, json.dumps(s, indent=2))
        lines = [f"Run fingerprint: {sig}   finished {s['finished']}",
                 f"Complete: {'YES' if complete else 'NO (re-run the same cell to resume)'}",
                 f"Photos: {len(df)} done, {n_failed} failed, {n_expected} expected",
                 f"Accepted (text and visual agree, IoU >= {AGREE_IOU}): {s['accepted']} ({s['accepted_pct']}%)   by split: {by_split}",
                 f"For review: {s['review']}   reasons: {s['review_reasons']}",
                 f"Median IoU between text and visual boxes: {s['median_iou_text_visual']}", "",
                 "Classes needing the most review:"] + [f"  {c}: {v:.0%}" for c, v in per_cls.head(15).items()]
        atomic_write_text(out_txt, "\n".join(lines)); print("\n".join(lines))
        return s


    # ==========================================================================================================
    # Main
    # ==========================================================================================================
    def main(limit=0):
        import types as _t
        args = _t.SimpleNamespace(**dict(locals()))
        tag = "_test" if args.limit else ""
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        log_path = start_logging(OUT_DIR, tag)
        out_csv, meta = OUT_DIR / f"boxes_visual{tag}.csv", OUT_DIR / f"boxes_visual{tag}.meta.json"
        failed_csv, progress = OUT_DIR / f"failed_photos_visual{tag}.csv", OUT_DIR / f"visual_progress{tag}.json"
        try:
            step(f"Step 1b: visual-prompt boxes ({SCRIPT_VERSION})"); print(f"Log file: {log_path}")
            import torch, transformers
            from transformers import Owlv2Processor, Owlv2ForObjectDetection
            print(f"Python {platform.python_version()} | torch {torch.__version__} | transformers {transformers.__version__} | "
                  f"GPU {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

            step("Inputs")
            for p in (DATA_ROOT, CLEAN_LIST, SPLIT_LIST, TEXT_BOXES):
                if not p.exists(): raise FileNotFoundError(f"Missing input: {p}" + ("  (run step 1 first)" if p == TEXT_BOXES else ""))
            text = pd.read_csv(TEXT_BOXES, dtype={"flag": str}); text["flag"] = text.flag.fillna("")
            split = pd.read_csv(SPLIT_LIST)
            clean = pd.read_csv(CLEAN_LIST); clean = clean[~clean.fname.isin(EXCLUDE)]
            clean["path"] = [str(DATA_ROOT / f"images_{c}" / f) for c, f in zip(clean.class_name, clean.fname)]
            photos = clean.rename(columns={"class_name": "object_id"}).merge(split[["fname", "split"]], on="fname", how="left")
            print(f"Photos: {len(photos)} | text boxes: {len(text)} | splits: {photos.split.value_counts().to_dict()}")
            if photos.split.isna().any(): raise ValueError(f"{int(photos.split.isna().sum())} photos have no split in {SPLIT_LIST}")

            step("Exemplars (training photos only)")
            ex = choose_exemplars(text, split)
            ex, manual_cls = apply_manual_exemplars(ex, photos)
            if manual_cls: print(f"Hand-drawn exemplars used for: {manual_cls}")
            missing = sorted(set(photos.object_id) - set(ex.object_id))
            few = ex.object_id.value_counts(); few = sorted(few[few < N_EXEMPLARS].index)
            print(f"Exemplars: {len(ex)} for {ex.object_id.nunique()} classes | classes with fewer than {N_EXEMPLARS}: {few} | none: {missing}")
            if missing: print("!! Classes without any exemplar are skipped (no confident training box). Box 2-3 of their photos by hand.")
            atomic_to_csv(ex, OUT_DIR / "exemplars.csv")
            try:
                obj_names = dict(pd.read_csv(OUT_DIR.parent / "object_list.csv", dtype=str).pipe(
                    lambda t: zip(t[[c for c in t.columns if t[c].astype(str).str.strip().str.match(r"OBJ\d+$").mean() > 0.5][0]].str.strip(),
                                  t["OBJECT_NAME"] if "OBJECT_NAME" in t.columns else t.iloc[:, 0])))
            except Exception: obj_names = None
            pages = exemplar_sheet(ex, OUT_DIR / "exemplar_sheet.jpg", obj_names, manual_cls)
            print(f"Saved exemplars.csv and {pages} (CHECK: every crop must show the right object)")
            crops = {c: [crop_exemplar(r) for _, r in g.iterrows()] for c, g in ex.groupby("object_id")}
            names = {c: list(g.fname) for c, g in ex.groupby("object_id")}
            photos = photos[photos.object_id.isin(crops)].reset_index(drop=True)
            if args.limit: photos = photos.sample(n=min(args.limit, len(photos)), random_state=0).reset_index(drop=True)

            step("Checkpoint")
            key = dict(version=SCRIPT_VERSION, model=MODEL_ID, n_ex=N_EXEMPLARS, ex_min=EXEMPLAR_MINSC, pad=EXEMPLAR_PAD,
                       text=file_md5(TEXT_BOXES), clean=file_md5(CLEAN_LIST), split=file_md5(SPLIT_LIST),
                       exemplars=sorted(ex.fname), exemplar_boxes=[round(float(v), 1) for v in ex[["x1", "y1", "x2", "y2"]].values.ravel()],
                       n=len(photos), limit=args.limit)
            sig = hashlib.md5(json.dumps(key, sort_keys=True).encode()).hexdigest()[:12]
            print(f"Run fingerprint: {sig}")
            prepare_checkpoint(out_csv, meta, sig, key)
            done = set(read_existing(out_csv).fname)
            todo = photos[~photos.fname.isin(done)].reset_index(drop=True)
            print(f"Already done: {len(done)} | to process now: {len(todo)}")

            elapsed = 0.0
            if len(todo):
                step(f"Loading {MODEL_ID}")
                device = "cuda" if torch.cuda.is_available() else "cpu"
                processor = Owlv2Processor.from_pretrained(MODEL_ID)
                model = Owlv2ForObjectDetection.from_pretrained(MODEL_ID).to(device).eval()
                if device == "cuda": model = model.half()

                dtype = torch.float16 if device == "cuda" else torch.float32
                K = N_EXEMPLARS

                def pixels(imgs):
                    return processor(images=imgs, return_tensors="pt")["pixel_values"].to(device, dtype)

                def flat(fmap):                                  # (B, H, W, D) -> (B, H*W, D)
                    b, h, w, d = fmap.shape
                    return fmap.reshape(b, h * w, d)

                @torch.no_grad()
                def fast_setup():
                    """Embed every class's exemplar crops ONCE (OWLv2 building blocks). Returns {class: (K, D) tensor}."""
                    q = {}
                    for cls, imgs in crops.items():
                        fmap = model.image_embedder(pixel_values=pixels(imgs))[0]
                        emb = model.embed_image_query(flat(fmap), fmap)[0]          # one embedding per usable exemplar
                        emb = emb.reshape(-1, emb.shape[-1])                         # (n_used, D); OWLv2 may skip a crop
                        if emb.shape[0] == 0: raise RuntimeError(f"no usable exemplar embedding for {cls}")
                        if emb.shape[0] < K: emb = torch.cat([emb, emb[:1].repeat(K - emb.shape[0], 1)])  # pad to K
                        q[cls] = emb[:K]
                    return q

                @torch.no_grad()
                def predict_fast(targets, classes):
                    fmap = model.image_embedder(pixel_values=pixels(targets))[0]   # ONE pass per photo
                    feats = flat(fmap)
                    qe = torch.stack([qemb[c] for c in classes])                   # (B, K, D)
                    logits = model.class_predictor(feats, qe)[0]                   # (B, P, K)
                    boxes = model.box_predictor(feats, fmap)                       # (B, P, 4)
                    return logits.float().cpu().numpy(), boxes.float().cpu().numpy()

                @torch.no_grad()
                def predict_fallback(targets, classes):
                    """Original image-guided detection, in small chunks of (photo, exemplar) pairs to stay in GPU memory."""
                    all_l, all_b = [], []
                    for t, c in zip(targets, classes):
                        qs = crops[c]; lg, bx = [], None
                        for s0 in range(0, len(qs), FALLBACK_PAIR_CHUNK):
                            qq = qs[s0:s0 + FALLBACK_PAIR_CHUNK]
                            inp = processor(images=[t] * len(qq), query_images=qq, return_tensors="pt").to(device)
                            out = model.image_guided_detection(pixel_values=inp["pixel_values"].to(dtype),
                                                               query_pixel_values=inp["query_pixel_values"].to(dtype))
                            lg.append(out.logits.float().cpu().numpy()[:, :, 0].T)  # (P, chunk)
                            bx = out.target_pred_boxes.float().cpu().numpy()[0]
                        l = np.concatenate(lg, axis=1)
                        if l.shape[1] < K: l = np.concatenate([l, np.full((l.shape[0], K - l.shape[1]), -1e4)], axis=1)
                        all_l.append(l); all_b.append(bx)
                    return np.stack(all_l), np.stack(all_b)

                predict = predict_fallback
                try:
                    qemb = fast_setup()
                    probe_l, probe_b = predict_fast([crops[next(iter(crops))][0]], [next(iter(crops))])
                    assert probe_l.shape[-1] == K and probe_b.shape[-1] == 4, (probe_l.shape, probe_b.shape)
                    predict = predict_fast
                    print(f"Fast path ON: exemplars embedded once ({len(qemb)} classes); one pass per photo")
                    if args.limit:                               # test run: prove the fast path matches the original method
                        chk = todo.head(12); imgs = [load_224(p) for p in chk.path]; cls = list(chk.object_id)
                        lf, bf = predict_fast(imgs, cls); lo, bo = predict_fallback(imgs, cls)
                        ok_all = True
                        for k in range(len(chk)):
                            n = len(names[cls[k]])
                            bf_k, sf, qf = best_box(lf[k][:, :n], bf[k]); bo_k, so, qo = best_box(lo[k][:, :n], bo[k])
                            corr, gap, ok = same_scoring(lf[k][:, :n], lo[k][:, :n]); ok_all &= ok
                            print(f"  check {chk.fname.iloc[k]}: score pattern r={corr:.4f} | logit gap {gap:.3f} | "
                                  f"box IoU {iou(bf_k, bo_k):.3f} | {'same' if ok else 'DIFFERENT'}")
                        if not ok_all:
                            print("!! Fast path scores photos differently from the original method -> using the original method")
                            predict = predict_fallback
                        else:
                            print(f"Fast path matches the original method (same score pattern on {len(chk)} photos)")
                except Exception as e:
                    print(f"!! Fast path unavailable in this transformers version ({e!r}).")
                    print("!! Using the slower original method (about 3x slower, same results).")

                step("Searching every photo with its own object's exemplars")
                _, elapsed = process(todo, crops, names, predict, out_csv, failed_csv, progress)

            step("Combine with text boxes, summary, previews")
            vis = read_existing(out_csv); vis = vis[vis.fname.isin(set(photos.fname))]
            comb = combine(text, vis)
            atomic_to_csv(comb, OUT_DIR / f"boxes_combined{tag}.csv")
            rev = comb[comb.decision == "review"][["fname", "object_id", "split", "path", "x1", "y1", "x2", "y2", "vx1", "vy1",
                                                  "vx2", "vy2", "iou_text_visual", "reason"]]
            atomic_to_csv(rev, OUT_DIR / f"review_list{tag}.csv")
            n_failed = count_failed(failed_csv)
            summarise(comb, len(photos), n_failed, elapsed, sig, OUT_DIR / f"visual_summary{tag}.txt", OUT_DIR / f"visual_run_summary{tag}.json")
            save_previews(comb, OUT_DIR / f"visual_previews{tag}")
            print(f"\nOutputs in {OUT_DIR}\n[{now()}] DONE")
        except KeyboardInterrupt:
            print(f"\n[{now()}] Interrupted. Everything up to the last batch is saved; re-run the same cell to resume."); raise
        except Exception:
            print(f"\n[{now()}] FAILED:\n{traceback.format_exc()}")
            print("Results up to the last completed batch are saved; fix the error and re-run the same cell to resume."); raise

    import types as _t
    return _t.SimpleNamespace(**{k: v for k, v in locals().items() if not k.startswith('__')})

S1B = _stage1b()
print('S1B: stage 1b (visual prompt) ready')

## Stages 2-3: self-trained detector and voting
```
Milestone 2, stages 2 and 3: self-trained detector + voting between three independent box sources.

STAGE 2 (self-training)
  A. Training labels = every unflagged, plausible text-prompt box (stage 1); where the visual-prompt box (stage 1b)
     agrees (IoU >= AGREE_IOU) the two are averaged. (The box sheets showed visual boxes are often wrong.)
  B. A small YOLO is trained to find our 73 objects on single photos, using ONLY agreed TRAIN-split photos
     (agreed val-split photos are used only for early stopping). Test photos are never used, so the model's
     boxes on validation and test photos are an independent opinion, not a copy of their automatic labels.
  C. The trained model is run on every photo: its box for the photo's own Object ID, and the strongest
     other Object ID it sees (a hint that a photo may be in the wrong folder or not show its object).
STAGE 3 (voting)
  Three candidate boxes per photo: T (text), V (visual), S (self-trained).
     all three agree (pairwise IoU >= AGREE_IOU) -> "strong"  box = mean of the three
     two agree                                   -> "auto"    box = mean of the two
     no two agree / not enough boxes             -> "review"  best guess offered to the reviewer
     S sees another Object ID strongly and not its own -> "review_label" (possible wrong folder / no object)
  Nothing is final here: stage 4 (review tool) lets a person confirm or fix every photo.

OUTPUTS (milestone2_outputs/)
  selftrain/dataset/...           YOLO dataset of agreed single photos (224 x 224)
  selftrain/run/                  Ultralytics training run (weights, curves); resumable
  boxes_selftrain.csv             stage 2 boxes for every photo (resumable)
  boxes_auto_final.csv            all candidate boxes, IoUs, decision and suggested box per photo
  fuse_summary.txt / fuse_summary.json, fuse_previews/<ID>.jpg, logs/stage23_<time>.log
```

In [ ]:
def _stage23():
    import argparse, csv, json, math, os, shutil, sys, time, traceback
    from itertools import combinations
    from pathlib import Path

    import numpy as np
    import pandas as pd
    from PIL import Image, ImageDraw


    SCRIPT_VERSION = "m2-stage23-v2"
    AGREE_IOU   = 0.70          # two boxes "agree" at this overlap
    S_MIN_CONF  = 0.25          # self-trained box must be at least this confident to vote
    LABEL_ALERT = 0.60          # another Object ID this confident (and own < S_MIN_CONF) -> "review_label"
    MODEL_INIT  = "yolo26s.pt"  # COCO-pretrained YOLO26 small (falls back to yolov8s.pt if unavailable)
    TRAIN_ARGS  = dict(imgsz=320, epochs=40, patience=10, batch=32, workers=2, seed=42, deterministic=True,
                       cos_lr=True, mosaic=0.5, close_mosaic=5, mixup=0.0, degrees=0.0, fliplr=0.5, scale=0.3,
                       translate=0.1, cache="ram", plots=True, verbose=True)
    PRED_BATCH  = 64
    S_FIELDS = ["fname", "object_id", "sx1", "sy1", "sx2", "sy2", "sscore", "s_other_id", "s_other_conf"]


    # ==========================================================================================================
    # Stage 2A: agreed photos
    # ==========================================================================================================
    def load_sources():
        photos = C.photo_table()
        t = pd.read_csv(C.TEXT_BOXES, dtype={"flag": str})
        t = t[["fname", "x1", "y1", "x2", "y2", "score", "flag"]].rename(
            columns={"x1": "tx1", "y1": "ty1", "x2": "tx2", "y2": "ty2", "score": "tscore", "flag": "tflag"})
        v = pd.read_csv(C.VIS_BOXES)[["fname", "vx1", "vy1", "vx2", "vy2", "vscore", "is_exemplar"]]
        df = photos.merge(t, on="fname", how="left").merge(v, on="fname", how="left")
        df["tflag"] = df.tflag.fillna("").astype(str).replace("nan", "")
        return df

    def tbox(r): return (r.tx1, r.ty1, r.tx2, r.ty2)
    def vbox(r): return (r.vx1, r.vy1, r.vx2, r.vy2)
    def sbox(r): return (r.sx1, r.sy1, r.sx2, r.sy2)

    def agreed(df):
        """Training labels for the self-trained detector.
        The box sheets showed the text-prompt boxes are mostly correct while the visual-prompt boxes are often large
        background regions, so the visual box is NOT required: every UNFLAGGED, plausible text box is used. Where the
        visual box also agrees, the two are averaged. (Only training-split photos are used for training, see 2B.)"""
        ok = [(r.tflag == "") and C.plausible(tbox(r)) for r in df.itertuples()]
        out = df[ok].copy()
        tv = np.array([C.iou(tbox(r), vbox(r)) for r in out.itertuples()]) if len(out) else np.array([])
        both = (tv >= AGREE_IOU) if len(out) else np.array([], bool)
        for k in "x1", "y1", "x2", "y2":
            out[k] = np.where(both, (out[f"t{k}"] + out[f"v{k}"]) / 2, out[f"t{k}"])
        out["label_source"] = np.where(both, "text+visual", "text")
        return out


    # ==========================================================================================================
    # Stage 2B: YOLO dataset of agreed single photos (train = train split, val = val split; test never used)
    # ==========================================================================================================
    def yolo_line(cid, b):
        x1, y1, x2, y2 = b; S = C.IMG_SIZE
        return f"{cid} {(x1 + x2) / 2 / S:.6f} {(y1 + y2) / 2 / S:.6f} {(x2 - x1) / S:.6f} {(y2 - y1) / S:.6f}"

    def build_dataset(agr, classes, ds_dir, sig):
        marker = ds_dir / "DATASET_OK.json"
        if marker.exists() and json.loads(marker.read_text(encoding="utf-8")).get("sig") == sig:
            print(f"Dataset already built for fingerprint {sig}: reusing {ds_dir}"); return ds_dir / "data.yaml"
        if ds_dir.exists(): shutil.rmtree(ds_dir)
        cid = {c: i for i, c in enumerate(classes)}; counts = {}
        for split in ("train", "val"):
            (ds_dir / "images" / split).mkdir(parents=True); (ds_dir / "labels" / split).mkdir(parents=True)
            rows = agr[agr.split == split]
            for r in rows.itertuples():
                try: im = C.load_224(r.path)
                except Exception as e: print(f"  ! skip {r.fname}: {e!r}"); continue
                stem = Path(r.fname).stem
                im.save(ds_dir / "images" / split / f"{stem}.jpg", quality=95)
                (ds_dir / "labels" / split / f"{stem}.txt").write_text(yolo_line(cid[r.object_id], (r.x1, r.y1, r.x2, r.y2)) + "\n")
            counts[split] = len(rows)
        yaml = ds_dir / "data.yaml"
        yaml.write_text(f"path: {ds_dir.as_posix()}\ntrain: images/train\nval: images/val\nnames:\n"
                        + "".join(f"  {i}: {c}\n" for i, c in enumerate(classes)), encoding="utf-8")
        C.atomic_write_text(marker, json.dumps(dict(sig=sig, counts=counts, built=C.now()), indent=2))
        print(f"Dataset built: {counts}")
        return yaml


    # ==========================================================================================================
    # Stage 2C: training (resumable) and prediction on every photo (resumable)
    # ==========================================================================================================
    def train(yaml, run_root, sig, quick):
        from ultralytics import YOLO
        run_dir = run_root / "selftrain"; done = run_dir / "DONE.json"
        best, last = run_dir / "weights" / "best.pt", run_dir / "weights" / "last.pt"
        if done.exists() and best.exists() and json.loads(done.read_text(encoding="utf-8")).get("sig") == sig:
            print(f"Self-trained model already finished for fingerprint {sig}: {best}"); return best
        if run_dir.exists() and not (last.exists() and (run_dir / "SIG").exists()
                                     and (run_dir / "SIG").read_text(encoding="utf-8").strip() == sig):
            stale = run_root / "_stale" / C.now().replace(":", "").replace(" ", "_"); stale.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(run_dir), str(stale)); print(f"!! Old training run belongs to other data/settings -> moved to {stale}")
        t0 = time.time()
        if last.exists():
            print(f"Resuming training from {last}"); YOLO(str(last)).train(resume=True)
        else:
            args = dict(TRAIN_ARGS, epochs=3 if quick else TRAIN_ARGS["epochs"])
            try: model = YOLO(MODEL_INIT)
            except Exception as e:
                print(f"!! {MODEL_INIT} unavailable ({e!r}) -> using yolov8s.pt"); model = YOLO("yolov8s.pt")
            run_dir.mkdir(parents=True, exist_ok=True); (run_dir / "SIG").write_text(sig, encoding="utf-8")
            print(f"Training {MODEL_INIT} with {json.dumps(args)}")
            model.train(data=str(yaml), project=str(run_root), name="selftrain", exist_ok=True, **args)
        C.atomic_write_text(done, json.dumps(dict(sig=sig, finished=C.now(), minutes_this_session=round((time.time() - t0) / 60, 1)), indent=2))
        return best

    def read_s(path):
        if not path.exists() or path.stat().st_size == 0: return pd.DataFrame(columns=S_FIELDS)
        df = pd.read_csv(path, on_bad_lines="skip", engine="python")
        return df.dropna(subset=["fname"]).drop_duplicates("fname", keep="last")

    def parse_result(cls, conf, xyxy, own_idx, names):
        """From one photo's detections: best box of its own class, and the strongest other class."""
        cls, conf, xyxy = np.asarray(cls).astype(int), np.asarray(conf, float), np.asarray(xyxy, float)
        own = np.where(cls == own_idx)[0]; other = np.where(cls != own_idx)[0]
        box, score = (np.nan,) * 4, 0.0
        if len(own):
            k = own[np.argmax(conf[own])]; box, score = tuple(float(v) for v in xyxy[k]), float(conf[k])
        oid, oconf = "", 0.0
        if len(other):
            k = other[np.argmax(conf[other])]; oid, oconf = names[int(cls[k])], float(conf[k])
        return box, score, oid, oconf

    def predict_all(photos, weights_or_fn, classes, out_csv, quick):
        """weights_or_fn: path to weights, or a function(list_of_PIL) -> list of (cls, conf, xyxy) for testing."""
        done = set(read_s(out_csv).fname)
        todo = photos[~photos.fname.isin(done)]
        if quick: todo = todo.head(300)
        print(f"Self-trained predictions: {len(done)} already done, {len(todo)} to do")
        if callable(weights_or_fn): run = weights_or_fn
        else:
            from ultralytics import YOLO
            model = YOLO(str(weights_or_fn))
            def run(imgs):
                res = model.predict(imgs, imgsz=TRAIN_ARGS["imgsz"], conf=0.01, max_det=20, verbose=False)
                return [(r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.xyxy.cpu().numpy()) for r in res]
        cid = {c: i for i, c in enumerate(classes)}
        new = not out_csv.exists() or out_csv.stat().st_size == 0
        fh = open(out_csv, "a", newline="", encoding="utf-8"); w = csv.DictWriter(fh, fieldnames=S_FIELDS)
        if new: w.writeheader(); fh.flush()
        t0 = time.time()
        try:
            for s in range(0, len(todo), PRED_BATCH):
                batch = todo.iloc[s:s + PRED_BATCH]; imgs, rows = [], []
                for r in batch.itertuples():
                    try: imgs.append(C.load_224(r.path)); rows.append(r)
                    except Exception as e: print(f"  ! could not read {r.fname}: {e!r}")
                for r, (cls, conf, xyxy) in zip(rows, run(imgs) if imgs else []):
                    box, score, oid, oconf = parse_result(cls, conf, xyxy, cid[r.object_id], classes)
                    w.writerow(dict(fname=r.fname, object_id=r.object_id, sx1=box[0], sy1=box[1], sx2=box[2], sy2=box[3],
                                    sscore=round(score, 4), s_other_id=oid, s_other_conf=round(oconf, 4)))
                fh.flush(); os.fsync(fh.fileno())
                n = s + len(batch); rate = n / max(time.time() - t0, 1e-6)
                C.atomic_write_text(C.OUT_DIR / "stage23_progress.json", json.dumps(dict(predicted=n, of=len(todo),
                                    photos_per_s=round(rate, 1), eta_min=round((len(todo) - n) / max(rate, 1e-6) / 60, 1), updated=C.now())))
                if (n // PRED_BATCH) % 10 == 0 or n >= len(todo):
                    print(f"  [{C.now()}] {n}/{len(todo)} photos | {rate:.1f}/s")
        finally:
            fh.close()


    # ==========================================================================================================
    # Stage 3: voting
    # ==========================================================================================================
    def vote(r):
        cands = {"T": tbox(r), "V": vbox(r)}
        if not pd.isna(r.sscore) and r.sscore >= S_MIN_CONF: cands["S"] = sbox(r)
        names = [n for n, b in cands.items() if C.plausible(b)]
        agree = []
        if len(names) == 3 and all(C.iou(cands[a], cands[b]) >= AGREE_IOU for a, b in combinations(names, 2)):
            agree = names
        else:
            pairs = sorted(((C.iou(cands[a], cands[b]), (a, b)) for a, b in combinations(names, 2)), reverse=True)
            if pairs and pairs[0][0] >= AGREE_IOU: agree = list(pairs[0][1])
        if agree:
            box = tuple(float(np.mean([cands[n][k] for n in agree])) for k in range(4))
            decision = "strong" if len(agree) == 3 else "auto"
        else:
            pick = next((n for n in ("S", "V", "T") if n in names), None)     # best guess for the reviewer
            box = cands[pick] if pick else (np.nan,) * 4
            decision = "review"
        own = 0.0 if pd.isna(r.sscore) else r.sscore
        if (not pd.isna(r.s_other_conf)) and r.s_other_conf >= LABEL_ALERT and own < S_MIN_CONF:
            decision = "review_label"
        return box, decision, "+".join(agree)

    def fuse(df):
        out = df.copy()
        res = [vote(r) for r in out.itertuples()]
        out["x1"], out["y1"], out["x2"], out["y2"] = zip(*[tuple(round(v, 1) if not pd.isna(v) else np.nan for v in b) for b, _, _ in res])
        out["decision"] = [d for _, d, _ in res]; out["agree"] = [a for _, _, a in res]
        out["iou_tv"] = [C.iou(tbox(r), vbox(r)) for r in out.itertuples()]
        out["iou_ts"] = [C.iou(tbox(r), sbox(r)) for r in out.itertuples()]
        out["iou_vs"] = [C.iou(vbox(r), sbox(r)) for r in out.itertuples()]
        return out

    def summarise(fz, sig, n_agreed):
        dec = fz.decision.value_counts().to_dict()
        by_split = (fz.groupby("split").decision.value_counts(normalize=True).mul(100).round(1).unstack(fill_value=0)).to_dict("index")
        per_cls = fz.assign(r=fz.decision.str.startswith("review")).groupby("object_id").r.mean().sort_values(ascending=False)
        s = dict(sig=sig, finished=C.now(), photos=int(len(fz)), agreed_text_visual=int(n_agreed), decisions=dec,
                 decisions_pct_by_split=by_split, review_label=fz[fz.decision == "review_label"][["fname", "s_other_id", "s_other_conf"]].head(50).to_dict("records"),
                 classes_needing_most_review={k: round(float(v), 3) for k, v in per_cls.head(15).items()})
        C.atomic_write_text(C.OUT_DIR / "fuse_summary.json", json.dumps(s, indent=2, default=str))
        lines = [f"Run fingerprint: {sig}   finished {s['finished']}",
                 f"Photos: {len(fz)} | usable text labels (training split used to train the self-trained model): {n_agreed}",
                 "Decisions: " + ", ".join(f"{k} {v} ({100 * v / len(fz):.1f}%)" for k, v in dec.items()),
                 "By split (%): " + json.dumps(by_split), "",
                 "strong = text, visual and self-trained boxes all agree; auto = two agree; review = no agreement;",
                 "review_label = the self-trained model sees a different Object ID, not this one.", "",
                 "Classes needing the most review:"] + [f"  {c}: {v:.0%}" for c, v in per_cls.head(15).items()]
        C.atomic_write_text(C.OUT_DIR / "fuse_summary.txt", "\n".join(lines)); print("\n".join(lines))

    def previews(fz, out_dir, per_class=12):
        out_dir.mkdir(parents=True, exist_ok=True); colours = {"T": (30, 200, 60), "V": (220, 0, 220), "S": (30, 120, 255)}
        for cls, g in fz.groupby("object_id"):
            g = pd.concat([g[g.decision.str.startswith("review")], g[~g.decision.str.startswith("review")]]).head(per_class)
            cols, cell = 6, C.IMG_SIZE + 26
            sheet = Image.new("RGB", (cols * cell, math.ceil(len(g) / cols) * cell), "white"); d = ImageDraw.Draw(sheet)
            for k, r in enumerate(g.itertuples()):
                try: im = C.load_224(r.path)
                except Exception: continue
                dd = ImageDraw.Draw(im)
                for n, b in (("T", tbox(r)), ("V", vbox(r)), ("S", sbox(r))):
                    if C.plausible(b): dd.rectangle(list(b), outline=colours[n], width=2)
                x0, y0 = (k % cols) * cell, (k // cols) * cell
                sheet.paste(im, (x0 + 13, y0 + 22))
                d.text((x0 + 13, y0 + 4), f"{str(r.fname)[:-4]} {r.decision} {r.agree}",
                       fill=(200, 0, 0) if r.decision.startswith("review") else (0, 0, 0))
            C.atomic_save_image(sheet, out_dir / f"{cls}.jpg", quality=90)


    # ==========================================================================================================
    # Main
    # ==========================================================================================================
    def main(quick=False):
        import types as _t
        args = _t.SimpleNamespace(**dict(locals()))
        tag = "_quick" if args.quick else ""
        C.OUT_DIR.mkdir(parents=True, exist_ok=True)
        log = C.start_logging("stage23", tag)
        root = C.OUT_DIR / f"selftrain{tag}"; s_csv = C.OUT_DIR / f"boxes_selftrain{tag}.csv"
        try:
            C.step(f"Stages 2-3 ({SCRIPT_VERSION})"); print(f"Log file: {log}")
            import torch, ultralytics
            print(f"torch {torch.__version__} | ultralytics {ultralytics.__version__} | "
                  f"GPU {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
            for p in (C.CLEAN_LIST, C.SPLIT_LIST, C.TEXT_BOXES, C.VIS_BOXES):
                if not p.exists(): raise FileNotFoundError(f"Missing input {p} (run stages 1 and 1b first)")

            C.step("Stage 2A: training labels = unflagged text boxes (visual box averaged in where it agrees)")
            df = load_sources(); classes = sorted(df.object_id.unique())
            missing_t, missing_v = int(df.tx1.isna().sum()), int(df.vx1.isna().sum())
            print(f"Photos: {len(df)} | without text box: {missing_t} | without visual box: {missing_v}")
            agr = agreed(df)
            print(f"Usable labels: {len(agr)} ({100 * len(agr) / len(df):.1f}%) | by split: {agr.split.value_counts().to_dict()} | "
                  f"sources: {agr.label_source.value_counts().to_dict()}")
            thin = agr[agr.split == "train"].object_id.value_counts().reindex(classes, fill_value=0)
            print(f"Agreed training photos per class: min {thin.min()} | classes with < 15: {list(thin[thin < 15].index)}")

            sig = C.fingerprint(dict(v=SCRIPT_VERSION, agree=AGREE_IOU, args=TRAIN_ARGS, init=MODEL_INIT, quick=args.quick,
                                     text=C.file_md5(C.TEXT_BOXES), vis=C.file_md5(C.VIS_BOXES), split=C.file_md5(C.SPLIT_LIST),
                                     exclude=sorted(C.EXCLUDE)))
            print(f"Run fingerprint: {sig}")
            meta = s_csv.with_name(s_csv.name + ".meta.json")
            if s_csv.exists() and (not meta.exists() or json.loads(meta.read_text(encoding="utf-8")).get("sig") != sig):
                stale = C.OUT_DIR / "_stale" / C.now().replace(":", "").replace(" ", "_"); stale.mkdir(parents=True, exist_ok=True)
                shutil.move(str(s_csv), str(stale / s_csv.name)); print(f"!! Old self-trained predictions archived to {stale}")
            C.atomic_write_text(meta, json.dumps(dict(sig=sig, created=C.now())))

            C.step("Stage 2B: YOLO dataset of agreed single photos (train and val splits only)")
            yaml = build_dataset(agr, classes, root / "dataset", sig)

            C.step("Stage 2C: train the self-trained detector (resumable)")
            weights = train(yaml, root, sig, args.quick)

            C.step("Stage 2D: self-trained boxes for every photo (resumable)")
            predict_all(df[["fname", "object_id", "path"]], weights, classes, s_csv, args.quick)

            C.step("Stage 3: voting")
            s = read_s(s_csv)
            fz = fuse(df.merge(s.drop(columns=["object_id"]), on="fname", how="left" if not args.quick else "inner"))
            C.atomic_to_csv(fz, C.OUT_DIR / f"boxes_auto_final{tag}.csv")
            summarise(fz, sig, len(agr))
            previews(fz, C.OUT_DIR / f"fuse_previews{tag}")
            complete = len(s) >= len(df) or args.quick
            print(f"\nStage 2-3 {'COMPLETE' if complete else 'INCOMPLETE (re-run to resume)'}: boxes_auto_final{tag}.csv\n[{C.now()}] DONE")
        except KeyboardInterrupt:
            print(f"\n[{C.now()}] Interrupted. Everything finished so far is saved; re-run the same cell to resume."); raise
        except Exception:
            print(f"\n[{C.now()}] FAILED:\n{traceback.format_exc()}\nFix the error and re-run the same cell; finished work is reused."); raise

    import types as _t
    return _t.SimpleNamespace(**{k: v for k, v in locals().items() if not k.startswith('__')})

S23 = _stage23()
print('S23: stages 2-3 (self-trained detector + voting) ready')

## Stage 4: review tool (people)
```
Milestone 2, stage 4: human review of every box (local app, nothing is uploaded).

TWO MODES
  verify   A page of photos (default 24) with their automatic box (yellow). Click any photo whose box is WRONG
           (it turns red), then press ENTER to confirm the page. Unclicked photos are recorded as correct.
           Covers every photo whose automatic decision is "strong" or "auto".
  fix      One photo at a time, enlarged, with all candidate boxes:
              1 = text box (green)   2 = visual box (magenta)   3 = self-trained box (blue)
           The current choice is yellow. Drag with the mouse to draw a new box.
              ENTER = save current box     N = object not visible (exclude)     L = it is a different object (exclude)
              S = skip for now             B = back to the previous photo        Q / ESC = save and quit
           Covers photos marked "review"/"review_label" and every photo flagged as wrong in verify mode.

GOOD BOX RULE (agree on this before starting): tight around the VISIBLE part of the labelled object only;
one box per photo; if the object is cut off at the edge, box the visible part.

SPLITTING THE WORK: give each person a range of Object IDs, e.g.
    python m2_review_tool.py --reviewer SC --mode verify --classes OBJ001-OBJ018
    python m2_review_tool.py --reviewer SC --mode fix    --classes OBJ001-OBJ018
Every action is written at once to milestone2_outputs/review/review_<reviewer>.csv (flushed to disk), so the
tool can be closed at any time and continues where it stopped. Re-reviewing a photo later simply overrides it.
Logs: milestone2_outputs/logs/review_<reviewer>_<time>.log
```

In [ ]:
def _review():
    import argparse, csv, os, re, sys
    from datetime import datetime
    from pathlib import Path

    import numpy as np
    import pandas as pd


    AUTO_FILE = C.OUT_DIR / "boxes_auto_final.csv"
    FIELDS = ["time", "reviewer", "mode", "fname", "object_id", "status", "x1", "y1", "x2", "y2", "source"]
    STATUSES = {"ok", "flag", "fixed", "exclude_no_object", "exclude_wrong_object", "skip"}


    # ==========================================================================================================
    # Core logic (no GUI; testable)
    # ==========================================================================================================
    def parse_classes(spec, all_classes):
        if not spec: return list(all_classes)
        out = []
        for part in spec.split(","):
            part = part.strip().upper()
            m = re.fullmatch(r"OBJ(\d+)-OBJ(\d+)", part)
            if m:
                lo, hi = int(m.group(1)), int(m.group(2)); out += [c for c in all_classes if lo <= int(c[3:]) <= hi]
            elif part: out.append(part)
        unknown = sorted(set(out) - set(all_classes))
        if unknown: raise ValueError(f"Unknown Object IDs: {unknown}")
        return sorted(set(out))

    def read_actions(review_dir=None, latest=True):
        """All reviewers' actions. latest=True: only the latest action per photo (it wins); False: full history."""
        review_dir = Path(review_dir or C.REVIEW_DIR)
        files = sorted(review_dir.glob("review_*.csv")) if review_dir.exists() else []
        parts = []
        for f in files:
            if f.stat().st_size == 0: continue                      # an empty file holds no actions
            try: parts.append(pd.read_csv(f, on_bad_lines="skip", engine="python", dtype={"status": str}))
            except Exception as e: print(f"!! could not read {f}: {e!r}")
        if not parts: return pd.DataFrame(columns=FIELDS)
        a = pd.concat(parts, ignore_index=True).dropna(subset=["fname", "status"])
        a = a[a.status.isin(STATUSES)].sort_values("time", kind="stable")
        return a.drop_duplicates("fname", keep="last") if latest else a.reset_index(drop=True)

    def build_queue(auto, mode, classes, actions, splits=None, sample=0, redo=False):
        """Photos still to do in this mode for these classes (and, optionally, only these splits), in a stable order.
        sample > 0 (verify mode): a fixed RANDOM sample of the automatically accepted photos, to measure their accuracy."""
        a = auto[auto.object_id.isin(classes)].copy()
        if splits: a = a[a.split.isin(splits)]
        if sample and mode == "verify":
            pool = a[a.decision.isin(["strong", "auto"])]
            a = pool.sample(n=min(int(sample), len(pool)), random_state=42)
        if redo and mode == "verify":
            # show EVERY photo of these classes again, with its CURRENT box (a person's box wins over the automatic one)
            if len(actions):
                cur = actions[actions.status.isin(["ok", "fixed"])].dropna(subset=["x1"]).set_index("fname")
                for k in ("x1", "y1", "x2", "y2"):
                    a[k] = [float(cur.at[f, k]) if f in cur.index else v for f, v in zip(a.fname, a[k])]
            a = a.dropna(subset=["x1"])
            return a.sort_values(["object_id", "fname"]).reset_index(drop=True)
        last = actions.set_index("fname").status.to_dict() if len(actions) else {}
        if mode == "verify":
            q = a[a.decision.isin(["strong", "auto"]) & ~a.fname.isin(last)]
        else:
            needs = a.decision.isin(["review", "review_label"]) | a.x1.isna()
            flagged = a.fname.map(lambda f: last.get(f) in ("flag", "skip"))
            settled = a.fname.map(lambda f: last.get(f) in ("fixed", "exclude_no_object", "exclude_wrong_object", "ok"))
            q = a[(needs & ~settled) | flagged]
        return q.sort_values(["object_id", "fname"]).reset_index(drop=True)

    class Recorder:
        """Appends one row per action and forces it to disk, so nothing is lost if the tool or laptop stops."""
        def __init__(self, reviewer, mode, review_dir=None):
            self.reviewer, self.mode = reviewer, mode
            d = Path(review_dir or C.REVIEW_DIR); d.mkdir(parents=True, exist_ok=True)
            self.path = d / f"review_{reviewer}.csv"
            new = not self.path.exists() or self.path.stat().st_size == 0
            self.fh = open(self.path, "a", newline="", encoding="utf-8"); self.w = csv.DictWriter(self.fh, fieldnames=FIELDS)
            if new: self.w.writeheader(); self._sync()
            self.n = 0
        def _sync(self): self.fh.flush(); os.fsync(self.fh.fileno())
        def record(self, row, status, box=None, source=""):
            assert status in STATUSES, status
            b = [round(float(v), 1) for v in box] if box is not None and not any(pd.isna(v) for v in box) else ["", "", "", ""]
            self.w.writerow(dict(time=datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f"), reviewer=self.reviewer, mode=self.mode,
                                 fname=row["fname"], object_id=row["object_id"], status=status,
                                 x1=b[0], y1=b[1], x2=b[2], y2=b[3], source=source))
            self._sync(); self.n += 1
        def close(self): self.fh.close()

    EXEMPLAR_FILE = C.OUT_DIR / "exemplars_manual.csv"
    EX_FIELDS = ["time", "reviewer", "fname", "object_id", "x1", "y1", "x2", "y2"]
    EX_PER_CLASS = 3

    class ExemplarRecorder:
        """Exemplar mode: saved boxes go to exemplars_manual.csv (used by stage 1b); other keys are ignored."""
        def __init__(self, reviewer, path=None):
            self.reviewer, self.mode = reviewer, "exemplar"; self.path = Path(path or EXEMPLAR_FILE)
            self.path.parent.mkdir(parents=True, exist_ok=True)
            new = not self.path.exists() or self.path.stat().st_size == 0
            self.fh = open(self.path, "a", newline="", encoding="utf-8"); self.w = csv.DictWriter(self.fh, fieldnames=EX_FIELDS)
            if new: self.w.writeheader(); self.fh.flush(); os.fsync(self.fh.fileno())
            self.n = 0; self.count = exemplar_counts(self.path)
        def record(self, row, status, box=None, source=""):
            if status != "fixed" or box is None: return
            self.w.writerow(dict(time=datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f"), reviewer=self.reviewer, fname=row["fname"],
                                 object_id=row["object_id"], x1=round(box[0], 1), y1=round(box[1], 1), x2=round(box[2], 1), y2=round(box[3], 1)))
            self.fh.flush(); os.fsync(self.fh.fileno()); self.n += 1
            self.count[row["object_id"]] = self.count.get(row["object_id"], 0) + 1
        def close(self): self.fh.close()

    def exemplar_counts(path=None):
        path = Path(path or EXEMPLAR_FILE)
        if not path.exists() or path.stat().st_size == 0: return {}
        m = pd.read_csv(path).drop_duplicates("fname", keep="last")
        return m.object_id.value_counts().to_dict()

    def exemplar_queue(classes, per_class=12):
        """Training photos of the chosen classes; the photos stage 1 was most confident about come first."""
        ph = C.photo_table(); ph = ph[(ph.split == "train") & ph.object_id.isin(classes)]
        if C.TEXT_BOXES.exists():
            t = pd.read_csv(C.TEXT_BOXES)[["fname", "x1", "y1", "x2", "y2", "score"]].rename(
                columns={"x1": "tx1", "y1": "ty1", "x2": "tx2", "y2": "ty2", "score": "tscore"})
            ph = ph.merge(t, on="fname", how="left")
        else:
            ph = ph.assign(tx1=np.nan, ty1=np.nan, tx2=np.nan, ty2=np.nan, tscore=0.0)
        ph = ph.sort_values(["object_id", "tscore"], ascending=[True, False]).groupby("object_id").head(per_class)
        ph = ph.assign(x1=ph.tx1, y1=ph.ty1, x2=ph.tx2, y2=ph.ty2, decision="exemplar", s_other_id="", s_other_conf=0.0)
        return ph.reset_index(drop=True)

    def candidates(row):
        out = {}
        for key, pre in (("1", "t"), ("2", "v"), ("3", "s")):
            b = tuple(row.get(f"{pre}{k}", np.nan) for k in ("x1", "y1", "x2", "y2"))
            if C.plausible(b, 0.0005, 1.01): out[key] = b
        return out

    def clamp_box(x1, y1, x2, y2):
        x1, x2 = sorted((x1, x2)); y1, y2 = sorted((y1, y2)); S = C.IMG_SIZE
        return (max(0, x1), max(0, y1), min(S, x2), min(S, y2))


    # ==========================================================================================================
    # GUI (tkinter, part of the standard Python install)
    # ==========================================================================================================
    def make_root(tk):
        """Sharp rendering on high-DPI Windows laptops; returns the window and the usable screen size."""
        if os.name == "nt":
            try:
                import ctypes; ctypes.windll.shcore.SetProcessDpiAwareness(1)
            except Exception: pass
        root = tk.Tk()
        return root, root.winfo_screenwidth(), root.winfo_screenheight()

    def run_verify(queue, rec, per_page, thumb):
        import tkinter as tk
        from PIL import ImageTk, ImageDraw
        cols = 6; rows_n = math.ceil(per_page / cols); pad = 6
        root, sw, sh = make_root(tk)
        thumb = int(max(96, min(thumb, (sw - 40) / cols - pad, (sh - 140) / rows_n - pad - 16)))   # always fits the screen
        root.title(f"VERIFY  |  reviewer {rec.reviewer}  |  click WRONG boxes, ENTER = confirm page, B = back, Q = quit")
        cv = tk.Canvas(root, width=cols * (thumb + pad), height=rows_n * (thumb + pad + 16) + 30, bg="white"); cv.pack()
        state = dict(page=0, flags=set(), imgs=[], cells=[])
        pages = [queue.iloc[i:i + per_page] for i in range(0, len(queue), per_page)]
        if not pages: print("Nothing to verify for these classes."); root.destroy(); return

        def draw():
            cv.delete("all"); state["imgs"], state["cells"] = [], []
            page = pages[state["page"]]
            for k, (_, r) in enumerate(page.iterrows()):
                x0, y0 = (k % cols) * (thumb + pad), (k // cols) * (thumb + pad + 16)
                try: im = C.load_224(r.path).resize((thumb, thumb))
                except Exception: im = Image.new("RGB", (thumb, thumb), "grey")
                s = thumb / C.IMG_SIZE; d = ImageDraw.Draw(im)
                bad = r.fname in state["flags"]
                d.rectangle([r.x1 * s, r.y1 * s, r.x2 * s, r.y2 * s], outline=(255, 0, 0) if bad else (255, 220, 0), width=3)
                if bad: d.line([0, 0, thumb, thumb], fill=(255, 0, 0), width=3); d.line([0, thumb, thumb, 0], fill=(255, 0, 0), width=3)
                tkim = ImageTk.PhotoImage(im); state["imgs"].append(tkim)
                cv.create_image(x0, y0 + 16, image=tkim, anchor="nw")
                cv.create_text(x0 + 2, y0 + 2, text=f"{r.object_id} {r.fname[-7:-4]}", anchor="nw", fill="red" if bad else "black")
                state["cells"].append((x0, y0 + 16, x0 + thumb, y0 + 16 + thumb, r.fname))
            done = sum(len(p) for p in pages[:state["page"]])
            cv.create_text(4, rows_n * (thumb + pad + 16) + 14, anchor="w",
                           text=f"Page {state['page'] + 1}/{len(pages)}  |  photos {done + 1}-{done + len(page)} of {len(queue)}  |  "
                                f"flagged on this page: {len(state['flags'])}  |  recorded this session: {rec.n}")

        def click(e):
            for x1, y1, x2, y2, f in state["cells"]:
                if x1 <= e.x <= x2 and y1 <= e.y <= y2:
                    state["flags"] ^= {f}; draw(); return

        def confirm(_=None):
            for _, r in pages[state["page"]].iterrows():
                if r.fname in state["flags"]: rec.record(r, "flag")
                else: rec.record(r, "ok", (r.x1, r.y1, r.x2, r.y2), source=f"auto:{r.decision}")
            print(f"[{C.now()}] page {state['page'] + 1}/{len(pages)} confirmed ({len(state['flags'])} flagged)")
            state["flags"] = set()
            if state["page"] + 1 < len(pages): state["page"] += 1; draw()
            else: print("All pages verified."); root.destroy()

        def back(_=None):
            if state["page"] > 0: state["page"] -= 1; state["flags"] = set(); draw()   # re-confirming overrides old rows

        cv.bind("<Button-1>", click); root.bind("<Return>", confirm); root.bind("<KP_Enter>", confirm)
        root.bind("b", back); root.bind("q", lambda e: root.destroy()); root.bind("<Escape>", lambda e: root.destroy())
        draw(); root.mainloop()

    def run_fix(queue, rec, scale, skip_fn=None):
        import tkinter as tk
        from PIL import ImageTk, ImageDraw
        root, sw, sh = make_root(tk)
        scale = max(1.5, min(scale, (sh - 140) / C.IMG_SIZE, (sw - 320) / C.IMG_SIZE))           # always fits the screen
        S = int(C.IMG_SIZE * scale)
        root.title(f"FIX  |  reviewer {rec.reviewer}  |  1/2/3 pick box, drag = draw, ENTER save, N no object, L other object, S skip, B back, Q quit")
        cv = tk.Canvas(root, width=S + 260, height=S + 40, bg="white"); cv.pack()
        colours = {"1": "#1ec83c", "2": "#dc00dc", "3": "#1e78ff"}
        state = dict(i=0, cur=None, src="", drag=None, img=None)
        if not len(queue): print("Nothing to fix for these classes."); root.destroy(); return

        def load():
            r = queue.iloc[state["i"]]; c = candidates(r)
            sugg = (r.x1, r.y1, r.x2, r.y2)
            state["cur"], state["src"] = (sugg, f"auto:{r.decision}") if C.plausible(sugg, 0.0005, 1.01) else \
                ((c[min(c)], f"cand{min(c)}") if c else (None, ""))
            draw()

        def draw():
            r = queue.iloc[state["i"]]; cv.delete("all")
            try: im = C.load_224(r.path).resize((S, S))
            except Exception: im = Image.new("RGB", (S, S), "grey")
            d = ImageDraw.Draw(im)
            for k, b in candidates(r).items():
                d.rectangle([v * scale for v in b], outline=colours[k], width=2); d.text((b[0] * scale + 3, b[1] * scale + 2), k, fill=colours[k])
            if state["cur"] is not None: d.rectangle([v * scale for v in state["cur"]], outline=(255, 220, 0), width=4)
            state["img"] = ImageTk.PhotoImage(im); cv.create_image(0, 0, image=state["img"], anchor="nw")
            info = (f"{r.object_id}\n{r.fname}\nsplit: {r.split}\nauto decision: {r.decision}\n"
                    + (f"self-trained sees: {r.s_other_id} ({r.s_other_conf:.2f})\n" if isinstance(r.get("s_other_id"), str) and r.s_other_id else "")
                    + f"\nphoto {state['i'] + 1} of {len(queue)}\nrecorded: {rec.n}\n\ncurrent box: {state['src'] or 'none'}\n\n"
                      "1 text (green)\n2 visual (magenta)\n3 self-trained (blue)\ndrag = draw new box\n\nENTER save\nN object not visible\n"
                      "L different object\nS skip   B back\nQ quit")
            cv.create_text(S + 12, 10, text=info, anchor="nw", font=("Arial", 10))

        def nxt():
            j = state["i"] + 1
            while skip_fn and j < len(queue) and skip_fn(queue.iloc[j]): j += 1
            if j < len(queue): state["i"] = j; load()
            else: print("All photos in this queue are done."); root.destroy()

        def save(_=None):
            if state["cur"] is None: return
            rec.record(queue.iloc[state["i"]], "fixed", state["cur"], state["src"]); nxt()
        def mark(status):
            rec.record(queue.iloc[state["i"]], status); nxt()
        def pick(k):
            c = candidates(queue.iloc[state["i"]])
            if k in c: state["cur"], state["src"] = c[k], f"cand{k}"; draw()
        def press(e): state["drag"] = (e.x / scale, e.y / scale)
        def motion(e):
            if state["drag"]:
                state["cur"] = clamp_box(state["drag"][0], state["drag"][1], e.x / scale, e.y / scale); state["src"] = "drawn"; draw()
        def release(e):
            if state["drag"]:
                b = clamp_box(state["drag"][0], state["drag"][1], e.x / scale, e.y / scale); state["drag"] = None
                if (b[2] - b[0]) > 3 and (b[3] - b[1]) > 3: state["cur"], state["src"] = b, "drawn"
                draw()
        def back(_=None):
            if state["i"] > 0: state["i"] -= 1; load()

        cv.bind("<ButtonPress-1>", press); cv.bind("<B1-Motion>", motion); cv.bind("<ButtonRelease-1>", release)
        root.bind("<Return>", save); root.bind("<KP_Enter>", save)
        for k in "123": root.bind(k, lambda e, k=k: pick(k))
        root.bind("n", lambda e: mark("exclude_no_object")); root.bind("l", lambda e: mark("exclude_wrong_object"))
        root.bind("s", lambda e: mark("skip")); root.bind("b", back)
        root.bind("q", lambda e: root.destroy()); root.bind("<Escape>", lambda e: root.destroy())
        load(); root.mainloop()


    def main(reviewer, mode, classes="", per_page=24, thumb=180, scale=2.5, splits="", sample=0, redo=False):
        import types as _t
        args = _t.SimpleNamespace(**dict(locals()))
        if not re.fullmatch(r"[A-Za-z]{1,6}", args.reviewer): raise ValueError("reviewer must be your initials (letters only)")
        log = C.start_logging(f"review_{args.reviewer}_{args.mode}")
        print(f"[{C.now()}] Review tool | reviewer {args.reviewer} | mode {args.mode} | log {log}")
        if args.mode == "exemplar":
            all_cls = sorted(C.photo_table().object_id.unique())
            classes = parse_classes(args.classes, all_cls) if args.classes else []
            if not classes: raise ValueError("exemplar mode needs REVIEW_CLASSES, e.g. 'OBJ040,OBJ043'")
            rec = ExemplarRecorder(args.reviewer.upper())
            queue = exemplar_queue(classes)
            queue = queue[queue.object_id.map(lambda c: rec.count.get(c, 0) < EX_PER_CLASS)].reset_index(drop=True)
            print(f"EXEMPLAR mode: draw a tight box on {EX_PER_CLASS} clear training photos per class "
                  f"(S = skip an unclear photo). Classes: {classes} | photos offered: {len(queue)}")
            try:
                run_fix(queue, rec, args.scale, skip_fn=lambda r: rec.count.get(r.object_id, 0) >= EX_PER_CLASS)
            finally:
                rec.close(); print(f"[{C.now()}] Saved {rec.n} exemplar boxes to {rec.path}; per class now: "
                                   f"{ {c: rec.count.get(c, 0) for c in classes} }")
            return
        if not AUTO_FILE.exists(): raise FileNotFoundError(f"Missing {AUTO_FILE}: run stages 2-3 first")
        auto = pd.read_csv(AUTO_FILE, dtype={"s_other_id": str})
        # Paths are rebuilt from THIS computer's dataset folder, so a teammate can review on their own laptop.
        auto["path"] = [str(C.DATA_ROOT / f"images_{c}" / f) for c, f in zip(auto.object_id, auto.fname)]
        missing = sum(not os.path.exists(p) for p in auto.path)
        print(f"Dataset folder on this computer: {C.DATA_ROOT} | photos not found: {missing}")
        if missing > 0.05 * len(auto):
            raise FileNotFoundError(f"{missing} photos not found under {C.DATA_ROOT}: set PROJECT_DIR in the Configuration cell")
        classes = parse_classes(args.classes, sorted(auto.object_id.unique()))
        splits = [x.strip().lower() for x in str(args.splits).split(",") if x.strip()]
        bad = sorted(set(splits) - {"train", "val", "test"})
        if bad: raise ValueError(f"unknown split(s): {bad} (use train, val, test)")
        queue = build_queue(auto, args.mode, classes, read_actions(), splits or None, int(args.sample or 0), bool(args.redo))
        print(f"Classes: {classes[0]}..{classes[-1]} ({len(classes)}) | splits: {splits or 'all'} | photos to {args.mode}: {len(queue)}")
        rec = Recorder(args.reviewer.upper(), args.mode)
        try:
            if args.mode == "verify": run_verify(queue, rec, args.per_page, args.thumb)
            else: run_fix(queue, rec, args.scale)
        finally:
            rec.close(); print(f"[{C.now()}] Session ended: {rec.n} actions saved to {rec.path}")


    import math                                   # used by the verify grid
    from PIL import Image                         # used for placeholders when a photo cannot be read

    import types as _t
    return _t.SimpleNamespace(**{k: v for k, v in locals().items() if not k.startswith('__')})

RV = _review()
print('RV: stage 4 (review tool) ready')

## Stage 5: master box file
```
Milestone 2, stage 5: the final, verified box for every photo.

Combines boxes_auto_final.csv (stages 2-3) with every reviewer's actions (stage 4); the latest action per photo wins.
  status "verified"          automatic box confirmed by a person (verify mode)
  status "fixed"             box chosen or drawn by a person (fix mode)
  status "excluded"          person said: object not visible / a different object (photo is not used)
  status "pending_fix"       flagged as wrong or skipped, not fixed yet
  status "unreviewed_auto"   automatic box (strong/auto) nobody has checked yet
  status "unreviewed_none"   no agreement and not reviewed yet (no usable box)

OUTPUTS (milestone2_outputs/)
  boxes_master.csv            one row per photo: box (224x224 frame), status, source, reviewer
  excluded_photos.csv         photos people excluded, with the reason
  master_report.txt / .json   coverage per split and per class, and how accurate the automatic boxes were
                              (share of verify-mode photos flagged as wrong; IoU between automatic and fixed boxes)
```

In [ ]:
def _master():
    read_actions, AUTO_FILE = RV.read_actions, RV.AUTO_FILE   # from the review stage
    import argparse, json, sys, traceback

    import numpy as np
    import pandas as pd



    def build(auto, actions):
        a = actions.set_index("fname") if len(actions) else pd.DataFrame(columns=["status"]).set_index(pd.Index([], name="fname"))
        rows = []
        for r in auto.itertuples():
            act = a.loc[r.fname] if r.fname in a.index else None
            st = None if act is None else act.status
            auto_box = (r.x1, r.y1, r.x2, r.y2)
            if st == "ok":
                box, status, source = (act.x1, act.y1, act.x2, act.y2), "verified", f"auto:{r.decision}"
            elif st == "fixed":
                box, status, source = (act.x1, act.y1, act.x2, act.y2), "fixed", str(act.source)
            elif st in ("exclude_no_object", "exclude_wrong_object"):
                box, status, source = (np.nan,) * 4, "excluded", st
            elif st in ("flag", "skip"):
                box, status, source = (np.nan,) * 4, "pending_fix", st
            elif r.decision in ("strong", "auto") and C.plausible(auto_box):
                box, status, source = auto_box, "unreviewed_auto", f"auto:{r.decision}"
            else:
                box, status, source = (np.nan,) * 4, "unreviewed_none", f"auto:{r.decision}"
            rows.append(dict(fname=r.fname, object_id=r.object_id, split=r.split, path=r.path,
                             x1=box[0], y1=box[1], x2=box[2], y2=box[3], status=status, source=source,
                             reviewer="" if act is None else act.reviewer, auto_decision=r.decision,
                             auto_x1=r.x1, auto_y1=r.y1, auto_x2=r.x2, auto_y2=r.y2))
        return pd.DataFrame(rows)

    def accuracy_of_auto(m, history):
        """How good were the automatic boxes? Measured on what people actually checked in verify mode.
        Uses the FULL history (a photo flagged in verify mode and fixed later still counts as flagged)."""
        out = {}
        vrows = history[history["mode"] == "verify"].drop_duplicates("fname", keep="last") if len(history) else history
        if len(vrows):
            dec = m.set_index("fname").auto_decision
            v = vrows.assign(decision=vrows.fname.map(dec))
            out["verify_checked"] = int(len(v))
            out["verify_flagged_pct"] = round(100 * float((v.status == "flag").mean()), 2)
            out["verify_flagged_pct_by_decision"] = {k: round(100 * float((g.status == "flag").mean()), 2) for k, g in v.groupby("decision")}
            sp = m.set_index("fname").split
            v = v.assign(split=v.fname.map(sp))
            out["verify_by_split"] = {k: dict(checked=int(len(g)), flagged=int((g.status == "flag").sum()),
                                              correct_pct=round(100 * float((g.status != "flag").mean()), 2)) for k, g in v.groupby("split")}
        fx = m[(m.status == "fixed")]
        if len(fx):
            ious = [C.iou((r.x1, r.y1, r.x2, r.y2), (r.auto_x1, r.auto_y1, r.auto_x2, r.auto_y2)) for r in fx.itertuples()]
            ious = [i for i in ious if not np.isnan(i)]
            out["fixed_photos"] = int(len(fx))
            if ious:
                out["iou_auto_vs_fixed_median"] = round(float(np.median(ious)), 3)
                out["fixed_kept_auto_box_pct"] = round(100 * float(np.mean(np.array(ious) >= 0.9)), 1)
        return out

    def main(strict=False, splits=""):
        import types as _t
        args = _t.SimpleNamespace(**dict(locals()))
        log = C.start_logging("stage5_master")
        try:
            C.step("Stage 5: master box file"); print(f"Log file: {log}")
            if not AUTO_FILE.exists(): raise FileNotFoundError(f"Missing {AUTO_FILE}: run m2_selftrain_fuse.py first")
            auto = pd.read_csv(AUTO_FILE, dtype={"s_other_id": str})
            actions = read_actions()
            print(f"Photos: {len(auto)} | review actions (latest per photo): {len(actions)} from "
                  f"{actions.reviewer.nunique() if len(actions) else 0} reviewer(s)")
            m = build(auto, actions)
            C.atomic_to_csv(m.drop(columns=["auto_x1", "auto_y1", "auto_x2", "auto_y2"]), C.OUT_DIR / "boxes_master.csv")
            C.atomic_to_csv(m[m.status == "excluded"][["fname", "object_id", "split", "source", "reviewer"]], C.OUT_DIR / "excluded_photos.csv")

            cov = m.groupby("split").status.value_counts().unstack(fill_value=0)
            final_ok = m.status.isin(["verified", "fixed", "excluded"])
            per_cls_left = m[~final_ok].object_id.value_counts()
            acc = accuracy_of_auto(m, read_actions(latest=False))
            complete = bool(final_ok.all())
            rep = dict(finished=C.now(), photos=int(len(m)), complete_human_checked=complete,
                       status_counts=m.status.value_counts().to_dict(), coverage_by_split=cov.to_dict("index"),
                       classes_with_most_left={k: int(v) for k, v in per_cls_left.head(20).items()},
                       usable_boxes=int(m.status.isin(["verified", "fixed"]).sum()), auto_accuracy=acc)
            C.atomic_write_text(C.OUT_DIR / "master_report.json", json.dumps(rep, indent=2, default=str))
            lines = [f"Master box file built {rep['finished']}",
                     f"Every photo checked by a person: {'YES' if complete else 'NO'}",
                     f"Status counts: {rep['status_counts']}",
                     f"Usable boxes (verified + fixed): {rep['usable_boxes']} of {len(m)}", "",
                     "Coverage by split:", cov.to_string(), "",
                     "Automatic-box accuracy (from the reviews): " + json.dumps(acc), "",
                     "Classes with the most photos left to review: " + json.dumps(rep["classes_with_most_left"])]
            C.atomic_write_text(C.OUT_DIR / "master_report.txt", "\n".join(lines)); print("\n".join(lines))
            chk = [x.strip().lower() for x in str(args.splits).split(",") if x.strip()]
            scope = m.split.isin(chk) if chk else pd.Series(True, index=m.index)
            left = int((~final_ok & scope).sum())
            print(f"Strict-check scope: {chk or 'all splits'} | photos not yet checked by a person in scope: {left}")
            if args.strict and left:
                print(f"\nSTRICT CHECK FAILED: {left} photos in {chk or 'all splits'} are not yet verified, fixed or excluded."); raise RuntimeError("STRICT CHECK FAILED")
            print(f"\n[{C.now()}] DONE: {C.OUT_DIR / 'boxes_master.csv'}")
        except SystemExit: raise
        except Exception:
            print(f"\n[{C.now()}] FAILED:\n{traceback.format_exc()}"); raise

    import types as _t
    return _t.SimpleNamespace(**{k: v for k, v in locals().items() if not k.startswith('__')})

MS = _master()
print('MS: stage 5 (master file) ready')

## Box viewing sheets (read-only)
One sheet per class showing **every** photo with all boxes found so far: green = text, magenta = visual, blue = self-trained, thick yellow = verified final box. Labels in red need attention. Run with step `sheets`; it never changes any box. Optional: set `M2_SHEET_CLASSES` (e.g. `OBJ004,OBJ020`) to draw only some classes.

In [ ]:
def _viewer():
    import math
    import numpy as np
    import pandas as pd
    from PIL import Image, ImageDraw

    SHEET_DIR = C.OUT_DIR / "box_sheets"
    COLS, THUMB = 10, 150
    COLOURS = {"T": (30, 200, 60), "V": (220, 0, 220), "S": (30, 120, 255), "M": (255, 210, 0)}

    def _boxes(df, pre):
        cols = [f"{pre}{k}" for k in ("x1", "y1", "x2", "y2")]
        return list(zip(*(df[c] for c in cols))) if all(c in df.columns for c in cols) else [None] * len(df)

    def gather():
        """Every photo with whatever boxes exist so far (text, visual, self-trained, verified master)."""
        df = C.photo_table()
        if C.TEXT_BOXES.exists():
            t = pd.read_csv(C.TEXT_BOXES)[["fname", "x1", "y1", "x2", "y2"]].rename(columns={"x1": "tx1", "y1": "ty1", "x2": "tx2", "y2": "ty2"})
            df = df.merge(t, on="fname", how="left")
        if C.VIS_BOXES.exists():
            df = df.merge(pd.read_csv(C.VIS_BOXES)[["fname", "vx1", "vy1", "vx2", "vy2"]], on="fname", how="left")
        s_csv = C.OUT_DIR / "boxes_selftrain.csv"
        if s_csv.exists() and s_csv.stat().st_size > 0:
            s = pd.read_csv(s_csv)[["fname", "sx1", "sy1", "sx2", "sy2", "sscore"]]
            df = df.merge(s, on="fname", how="left")
        f_csv = C.OUT_DIR / "boxes_auto_final.csv"
        if f_csv.exists(): df = df.merge(pd.read_csv(f_csv)[["fname", "decision"]], on="fname", how="left")
        m_csv = C.OUT_DIR / "boxes_master.csv"
        if m_csv.exists():
            m = pd.read_csv(m_csv)[["fname", "x1", "y1", "x2", "y2", "status"]].rename(columns={"x1": "mx1", "y1": "my1", "x2": "mx2", "y2": "my2"})
            df = df.merge(m, on="fname", how="left")
        return df

    def label(r):
        if "status" in r and isinstance(r["status"], str) and r["status"] in ("verified", "fixed", "excluded"):
            return r["status"].upper(), r["status"] == "excluded"
        if "decision" in r and isinstance(r["decision"], str):
            return r["decision"].upper(), r["decision"].startswith("review")
        t, v = (r.get("tx1"), r.get("ty1"), r.get("tx2"), r.get("ty2")), (r.get("vx1"), r.get("vy1"), r.get("vx2"), r.get("vy2"))
        i = C.iou(t, v)
        if np.isnan(i): return ("TEXT ONLY" if not pd.isna(t[0]) else "NO BOX"), True
        return (f"AGREE {i:.2f}" if i >= 0.7 else f"DIFFER {i:.2f}"), i < 0.7

    def sheet(cls, g):
        g = g.sort_values("fname").reset_index(drop=True)
        rows = math.ceil(len(g) / COLS); cell_h = THUMB + 22; head = 40
        im = Image.new("RGB", (COLS * (THUMB + 6) + 6, head + rows * cell_h), "white"); d = ImageDraw.Draw(im)
        n_bad = 0; s = THUMB / C.IMG_SIZE
        for k, r in g.iterrows():
            x0, y0 = 6 + (k % COLS) * (THUMB + 6), head + (k // COLS) * cell_h
            try: ph = C.load_224(r["path"]).resize((THUMB, THUMB))
            except Exception: ph = Image.new("RGB", (THUMB, THUMB), "grey")
            dd = ImageDraw.Draw(ph)
            for key, pre, w in (("T", "t", 2), ("V", "v", 2), ("S", "s", 2), ("M", "m", 4)):
                b = tuple(r.get(f"{pre}{c}") for c in ("x1", "y1", "x2", "y2"))
                if not any(v is None or pd.isna(v) for v in b): dd.rectangle([v * s for v in b], outline=COLOURS[key], width=w)
            im.paste(ph, (x0, y0))
            txt, bad = label(r); n_bad += bad
            d.text((x0, y0 + THUMB + 3), f"{str(r['fname'])[-7:-4]} {txt}", fill=(200, 0, 0) if bad else (0, 0, 0))
        d.text((6, 6), f"{cls}  |  {len(g)} photos  |  needs attention: {n_bad}", fill=(0, 0, 0))
        d.text((6, 22), "green = text   magenta = visual   blue = self-trained   thick yellow = verified final box", fill=(90, 90, 90))
        C.atomic_save_image(im, SHEET_DIR / f"{cls}.jpg", quality=88)
        return n_bad

    def main(classes=""):
        log = C.start_logging("sheets")
        C.step("Box viewing sheets (read-only)"); print(f"Log file: {log}")
        SHEET_DIR.mkdir(parents=True, exist_ok=True)
        df = gather()
        want = [c.strip().upper() for c in classes.split(",") if c.strip()] or sorted(df.object_id.unique())
        sources = [n for n, c in (("text", "tx1"), ("visual", "vx1"), ("self-trained", "sx1"), ("verified", "mx1")) if c in df.columns]
        print(f"Box sources found: {sources} | classes: {len(want)}")
        summary = {}
        for cls in want:
            summary[cls] = sheet(cls, df[df.object_id == cls])
        worst = sorted(summary.items(), key=lambda kv: -kv[1])[:15]
        lines = [f"Sheets written to {SHEET_DIR} ({len(summary)} classes, sources: {sources})",
                 "Classes with the most photos needing attention: " + ", ".join(f"{c} ({n})" for c, n in worst)]
        C.atomic_write_text(SHEET_DIR / "README.txt", "\n".join(lines)); print("\n".join(lines))

    import types as _t
    return _t.SimpleNamespace(**{k: v for k, v in locals().items() if not k.startswith('__')})

VW = _viewer()
print('VW: box viewing sheets ready')

# Run
Each cell below runs only if its step is in `RUN`. Order: `stage1_test` → check → `stage1` → `stage1b_test` → check the exemplar sheet → `stage1b` → `stage23_quick` → `stage23` → people review → `master` / `master_strict`.

In [ ]:
def run_step(name, fn, **kw):
    """Run one step if it was requested; logging is always closed afterwards, errors stop the notebook."""
    if name not in RUN:
        print(f"[skip] {name}"); return
    print(f"[run ] {name} {kw}")
    try:
        fn(**kw)
    finally:
        C.stop_logging()

In [ ]:
run_step("stage1_test", S1.main, limit=50)
run_step("stage1", S1.main)

In [ ]:
run_step("stage1b_test", S1B.main, limit=60)
run_step("stage1b", S1B.main)

In [ ]:
run_step("sheets", VW.main, classes=os.environ.get("M2_SHEET_CLASSES", ""))

In [ ]:
run_step("stage23_quick", S23.main, quick=True)
run_step("stage23", S23.main)

## Review (people, in VS Code or Jupyter)
Set `REVIEWER`, `REVIEW_MODE` and `REVIEW_CLASSES` in the Configuration cell, run the cells from the top down to here (definitions are quick), then run this cell. A window opens; every action is saved at once.

**Good box rule:** tight around the visible part of the labelled object only; one box per photo; if the object is cut off at the edge, box the visible part.

| Person | `REVIEW_CLASSES` |
|---|---|
| 1 | `OBJ001-OBJ018` |
| 2 | `OBJ019-OBJ036` |
| 3 | `OBJ037-OBJ054` |
| 4 | `OBJ055-OBJ073` |

In [ ]:
if REVIEWER:
    try:
        RV.main(reviewer=REVIEWER, mode=REVIEW_MODE, classes=REVIEW_CLASSES, splits=REVIEW_SPLITS, sample=REVIEW_SAMPLE, redo=REVIEW_REDO)
    finally:
        C.stop_logging()
else:
    print("[skip] review (REVIEWER is empty)")

In [ ]:
run_step("master", MS.main)
run_step("master_strict", MS.main, strict=True, splits=os.environ.get("M2_STRICT_SPLITS", "test"))